# Entorno de análisis — pronóstico de ventas cliente–línea

**Tesis:** *Modelos de predicción de ventas mediante técnicas de machine learning en una empresa retail dedicada a la distribución de productos farmacéuticos* — Jonathan Fernando Pallasco Catota (UDLA).

Este cuaderno ejecuta, en un solo flujo, todo lo que va **después** del entorno seguro (capítulos IV y V):

| Paso | Qué hace | Sección de la tesis |
|---|---|---|
| 0 | Rutas, parámetros y librerías | — |
| 1 | Carga la base sintética y las series externas; clasifica la demanda (ADI y CV²) | 4.2 |
| 2 | Pruebas estadísticas y figuras del análisis exploratorio | 4.3–4.4 |
| 3 | Conjunto de pronóstico directo multihorizonte (sin fuga de información) | 4.2 y 4.5 |
| 4 | Matriz de correlación y Figura 4 | 4.4 |
| 5 | Decisiones en validación: transformación de la venta, exógenas y ACP | 5.2 |
| 6 | Búsqueda de hiperparámetros, entrenamiento final y prueba de los cinco modelos | 5.2–5.3 |
| 7 | Evaluación: métricas, horizonte, tipo de relación, niveles de agregación y bootstrap | 5.3–5.4 |
| 8 | Curvas de aprendizaje y error por partición | 5.2 |
| 9 | Combinación Random Forest + Croston–SBA con intervalo del 80 % | 5.5 |
| 10 | Réplica del estudio de Supermercados Tía | 5.6 |
| 11 | Interpretabilidad: SHAP, LIME y permutación | 5.7 |
| 12 | Figuras del capítulo V y resumen de resultados | V y VI |
| 13 | Figuras y tablas de los anexos | Anexos 3 a 5 |

**Insumos** (los genera el entorno seguro): `Data/publico/base_sintetica.csv`, `Data/publico/exogenas_alineadas.csv` y `Data/publico/exogenas_admitidas.json`. Opcional: `Data/Seguro/outputs/publicable/tablas/C_validacion_sintetica_estudio.csv` (solo agregados) para la figura de validación de la base sintética.

**Salidas:** `Data/Analisis/outputs/figuras` (cada figura en SVG vectorial en `svg/` y en PNG a 300 ppp como vista previa), `Data/Analisis/outputs/tablas` (CSV) y `Data/Analisis/outputs/intermedios` (modelos y predicciones; no se publican).

**Librerías:** `pip install pandas numpy scipy scikit-learn statsmodels matplotlib xgboost lightgbm catboost shap lime holidays openpyxl`.

**Rehacer solo las figuras:** ejecutar los Pasos 0 a 4 y luego 12 y 13; leen de disco los modelos y resultados ya calculados (unos 2 minutos).

**Tiempo:** alrededor de 1,5 a 2 horas en un equipo de 2 núcleos. Con `MODO_PRUEBA = True` corre en pocos minutos sobre una muestra de relaciones (solo para verificar que todo funciona; no usar esas cifras en la tesis).

## Paso 0. Rutas y parámetros

In [ ]:
from pathlib import Path

RUTA = Path.cwd()
DIR_PUBLICO = RUTA / "Data/Publico"                                   # insumos del entorno seguro
DIR_TABLAS_SEGURO = RUTA / "Data/Seguro/outputs/publicable/tablas"    # opcional: validación de la base sintética
SALIDAS = RUTA / "Data/Analisis/outputs"
DIR_FIG = SALIDAS / "figuras"
DIR_TAB = SALIDAS / "tablas"
DIR_INT = SALIDAS / "intermedios"
SEMILLA = 20240601

# Período y particiones (por mes pronosticado)
PERIODO_INICIO = "2022-01"
PERIODO_FIN = "2026-09"
FIN_ENTRENAMIENTO = "2025-09"     # entrenamiento: meses pronosticados hasta aquí
FIN_VALIDACION = "2026-03"        # validación: 2025-10 a 2026-03; prueba: 2026-04 a 2026-09
MESES_HISTORIA_MIN = 6            # el método de la empresa necesita 6 meses previos
HORIZONTES = [1, 2, 3, 4, 5, 6]

# Protocolo de modelado
TRANSFORMACIONES = ["slog", "nivel"]        # slog = signo(y)·ln(1+|y|)
MAX_FILAS_BUSQUEDA = 200_000                # submuestra para RF y MLP en la búsqueda
N_BOOTSTRAP = 1000
N_SHAP, N_PERMUTACION = 3000, 15000
REJILLAS = {
    "Random Forest": {"n_estimators": [150], "min_samples_leaf": [5, 20], "max_features": [0.33, 0.6],
                      "max_samples": [0.5]},
    "MLP": {"capas": [(128, 64), (64, 32)], "alpha": [1e-4, 1e-3]},
    "XGBoost": {"max_depth": [6, 8], "min_child_weight": [5, 20]},
    "LightGBM": {"num_leaves": [31, 63], "min_child_samples": [20, 100]},
    "CatBoost": {"depth": [6, 8], "l2": [3, 10]},
}
MODELOS = list(REJILLAS)

# Verificación rápida (NO usar sus cifras en la tesis)
MODO_PRUEBA = False
SESGO_MAX_PCT = 20.0              # sesgo máximo admitido del total (validación) al elegir la transformación
N_RELACIONES_PRUEBA = 400

for nombre, ruta in [("base_sintetica.csv", DIR_PUBLICO / "base_sintetica.csv"),
                     ("exogenas_alineadas.csv", DIR_PUBLICO / "exogenas_alineadas.csv"),
                     ("exogenas_admitidas.json", DIR_PUBLICO / "exogenas_admitidas.json"),
                     ("validación sintética (opcional)", DIR_TABLAS_SEGURO / "C_validacion_sintetica_estudio.csv")]:
    print(f"{nombre:34s} {'OK' if ruta.exists() else 'NO ENCONTRADO'} -> {ruta}")

### Librerías, métricas y estilo de las figuras
Las figuras se dibujan a su tamaño de impresión (15 cm de ancho útil) con letra de 8 a 9 pt, sin título interno (la descripción va en el pie de figura de Word) y con una sola paleta: un color fijo por modelo y grises para los métodos de referencia.

In [ ]:
import gc
import itertools
import json
import logging
import pickle
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from IPython.display import Image, display

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

for d in [DIR_FIG, DIR_TAB, DIR_INT]:
    d.mkdir(parents=True, exist_ok=True)

CM = 1 / 2.54
ANCHO = 15 * CM                      # ancho útil de la página (márgenes de 3 cm)
# Paleta institucional sobria: azules y turquesa para los modelos, grises para las referencias y naranja
# solo para el contraste principal (el método actual de la empresa) y las alertas.
AZUL_MARINO, AZUL, TURQUESA, GRIS, GRIS_CLARO, NARANJA = "#17365D", "#34658A", "#2996A5", "#747D87", "#D9E0E5", "#C98042"
PALETA = [AZUL_MARINO, NARANJA, TURQUESA, AZUL, GRIS]            # 0 principal, 1 contraste/alerta, 2 secundario
GRISES = ["#4D5660", GRIS, "#B8C1C9"]
TINTA = "#1E2328"
COLOR_MODELO = {"Random Forest": AZUL_MARINO, "MLP": "#9AA5B1", "XGBoost": AZUL, "LightGBM": "#5FAAB8", "CatBoost": "#8EAFC8"}
COLOR_COMB = TURQUESA
COLOR_EMP = NARANJA
COLOR_BANDA = "#BFD3E1"
AZULES = LinearSegmentedColormap.from_list("azules", ["#FFFFFF", GRIS_CLARO, AZUL, AZUL_MARINO])
DIVERGENTE = LinearSegmentedColormap.from_list("div", [AZUL_MARINO, "#F2F4F6", NARANJA])
FONDO_PART = {"entrenamiento": "#F1F4F7", "validacion": "#E6F2F4", "prueba": "#F8EFE7"}
MES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
EMP, EMP_OR, CRO = "Método empresa", "Método empresa (línea real)", "Croston–SBA"
REF_ESTILO = {EMP: (COLOR_EMP, "--"), CRO: (GRISES[1], "-."), EMP_OR: (COLOR_EMP, ":"),
              "Media móvil 6": (GRISES[2], "--")}


def estilo():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "svg.fonttype": "none",
        "font.family": ["Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8.5, "axes.labelsize": 8.5, "axes.titlesize": 8.5,
        "xtick.labelsize": 7.8, "ytick.labelsize": 7.8, "legend.fontsize": 7.8,
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRIS,
        "axes.labelcolor": TINTA, "xtick.color": GRISES[0], "ytick.color": GRISES[0],
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.grid": True, "grid.color": GRIS_CLARO, "grid.linewidth": 0.5, "grid.alpha": 1.0,
        "legend.frameon": False, "lines.linewidth": 1.2, "lines.markersize": 3.5, "axes.linewidth": 0.6,
        "patch.linewidth": 0.5, "axes.prop_cycle": plt.cycler(color=[AZUL_MARINO, TURQUESA, AZUL, GRIS, NARANJA]),
    })


def etiqueta(ax, texto, x=0.0, y=1.02):
    ax.text(x, y, texto, transform=ax.transAxes, ha="left", va="bottom", fontsize=8.5,
            color=GRISES[0], fontweight="bold")


def _formato_es(fig):
    """Decimales con coma y signo menos tipográfico en los ejes numéricos (también en las barras de color)."""
    from matplotlib.ticker import ScalarFormatter, FuncFormatter
    for ax in fig.axes:
        for eje in (ax.xaxis, ax.yaxis):
            if type(eje.get_major_formatter()) is ScalarFormatter:
                eje.set_major_formatter(FuncFormatter(lambda v, p: f"{v:g}".replace(".", ",").replace("-", "−")))


DIR_SVG = DIR_FIG / "svg"
DIR_SVG.mkdir(parents=True, exist_ok=True)


def guardar(fig, nombre, mostrar=True):
    """Guarda la figura en SVG vectorial (figuras/svg, para la tesis) y en PNG a 300 ppp (figuras, vista previa)."""
    _formato_es(fig)
    destino = DIR_FIG / f"{nombre}.png"
    fig.savefig(destino, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(DIR_SVG / f"{nombre}.svg", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    if mostrar:
        display(Image(filename=str(destino), width=640))
    return destino


def coma(v, d=1):
    if abs(v) < 0.5 * 10 ** (-d):
        v = 0.0
    return f"{v:.{d}f}".replace(".", ",").replace("-", "−")


def metricas(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    e = y - p
    mse = float(np.mean(e ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    den = np.abs(y) + np.abs(p)
    smape = np.where(den == 0, 0.0, 2 * np.abs(e) / np.where(den == 0, 1, den))
    pos = y != 0
    return {"n": int(len(y)), "MAE": float(np.mean(np.abs(e))), "MSE": mse, "RMSE": float(np.sqrt(mse)),
            "R2": float(1 - np.sum(e ** 2) / sst) if sst > 0 else np.nan,
            "WAPE": float(np.sum(np.abs(e)) / np.sum(np.abs(y))) if np.sum(np.abs(y)) > 0 else np.nan,
            "sMAPE": float(np.mean(smape)),
            "MAPE_y_pos": float(np.mean(np.abs(e[pos]) / np.abs(y[pos]))) if pos.any() else np.nan,
            "sesgo_total_pct": float(100 * (p.sum() - y.sum()) / abs(y.sum())) if y.sum() != 0 else np.nan}


def wape(y, p):
    return float(np.abs(np.asarray(y) - np.asarray(p)).sum() / np.abs(np.asarray(y)).sum())


def guardar_json(obj, ruta):
    def conv(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=conv)


estilo()
print("Salidas en", SALIDAS)

## Paso 1. Base sintética, series externas y tipo de demanda
Cada relación cliente–línea se clasifica con el intervalo medio entre compras (ADI) y el cuadrado del coeficiente de variación del monto (CV²), **solo con datos hasta septiembre de 2025** (Petropoulos et al., 2022), con los cortes 1,32 y 0,49.

In [ ]:
ATR_MAP = {"tipo_cliente": "tratamiento", "detalle_negocio": "detalle_neg", "provincia": "provincia",
           "region": "region"}
NOM_SEG = {"suave": "Suave", "errática": "Errática", "intermitente": "Intermitente",
           "irregular (lumpy)": "Irregular", "nueva/insuficiente": "Nueva"}


def clasificar_intermitencia(panel_, fin=FIN_ENTRENAMIENTO):
    h = panel_[panel_["mes"] <= pd.Period(fin, "M")]
    g = h.groupby("par_id")
    tam = h[h["activo"] == 1].assign(a=lambda d: d["ventas_usd"].abs()).groupby("par_id")["a"]
    est = pd.DataFrame({"n": g.size(), "k": g["activo"].sum(), "cv2": (tam.std(ddof=0) / tam.mean()) ** 2})
    est["adi"] = est["n"] / est["k"].replace(0, np.nan)

    def cls(r):
        if pd.isna(r.adi) or r.k < 2:
            return "nueva/insuficiente"
        if r.adi < 1.32:
            return "suave" if r.cv2 < 0.49 else "errática"
        return "intermitente" if r.cv2 < 0.49 else "irregular (lumpy)"
    est["segmento_sb"] = est.apply(cls, axis=1)
    panel_ = panel_.merge(est[["segmento_sb", "adi", "cv2"]], left_on="par_id", right_index=True, how="left")
    panel_["segmento_sb"] = panel_["segmento_sb"].fillna("nueva/insuficiente")
    return panel_


def cargar_sintetica():
    d = pd.read_csv(DIR_PUBLICO / "base_sintetica.csv", dtype={"cliente": str, "linea": str})
    d["mes"] = pd.PeriodIndex(d["mes"], freq="M")
    if MODO_PRUEBA:
        rel = d[["cliente", "linea"]].drop_duplicates().sample(N_RELACIONES_PRUEBA, random_state=SEMILLA)
        d = d.merge(rel, on=["cliente", "linea"])
    d = d.sort_values(["cliente", "linea", "mes"]).reset_index(drop=True)
    d["par_id"] = d.groupby(["cliente", "linea"], sort=True).ngroup()
    d["activo"] = (d["venta_neta"] != 0).astype(int)
    d["primer_mes"] = d.groupby("par_id")["mes"].transform("min")
    p = d.rename(columns={"linea": "sku", "venta_neta": "ventas_usd"})
    atr = p.groupby("par_id")[list(ATR_MAP)].first().rename(columns=ATR_MAP)
    atr["linea"] = p.groupby("par_id")["sku"].first()
    p = clasificar_intermitencia(p.drop(columns=list(ATR_MAP)))
    return p, atr


panel, atributos = cargar_sintetica()
exog = pd.read_csv(DIR_PUBLICO / "exogenas_alineadas.csv", index_col=0)
exog.index = pd.PeriodIndex(exog.index, freq="M")
ADMITIDAS = json.load(open(DIR_PUBLICO / "exogenas_admitidas.json", encoding="utf-8"))["admitidas"]
TODAS_MACRO = ["ipc_general_var_a", "ipc_salud_var_a", "desempleo", "imae_var_a", "riesgo_pais", "tasa_activa",
               "remesas_var_a", "dengue_log", "petroleo_wti", "oni"]
seg_par = panel.groupby("par_id")["segmento_sb"].first()
act = panel[panel.activo == 1]
resumen_datos = {
    "filas_relacion_mes": len(panel), "relaciones": int(panel.par_id.nunique()),
    "clientes": int(panel.cliente.nunique()), "lineas": int(panel.sku.nunique()),
    "pct_meses_sin_compra": 100 * (1 - panel.activo.mean()),
    "tipos_de_relacion": seg_par.value_counts().to_dict(),
    "mediana_usd_meses_con_movimiento": float(act.ventas_usd.abs().median()),
    "p90_usd_meses_con_movimiento": float(act.ventas_usd.abs().quantile(0.9)),
    "pct_negativos_en_meses_con_movimiento": float(100 * (act.ventas_usd < 0).mean()),
    "series_externas_admitidas": ADMITIDAS,
}
guardar_json(resumen_datos, DIR_TAB / "R01_resumen_datos.json")
pd.Series(resumen_datos).to_frame("valor")

## Paso 2. Naturaleza de los datos y análisis exploratorio
Pruebas sobre la serie agregada de la empresa: ADF y KPSS (estacionariedad), tendencia lineal, Ljung–Box (autocorrelación), fuerza de tendencia y estacionalidad con STL, y BDS sobre los residuos de un AR(1) (linealidad).

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss, bds
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm


def pruebas_serie(y, nombre):
    y = pd.Series(np.asarray(y, float))
    t = np.arange(len(y))
    f = []
    a = adfuller(y, autolag="AIC")
    f.append(("ADF (H0: raíz unitaria)", a[0], a[1]))
    k = kpss(y, regression="c", nlags="auto")
    f.append(("KPSS nivel (H0: estacionaria)", k[0], k[1]))
    kt = kpss(y, regression="ct", nlags="auto")
    f.append(("KPSS tendencia (H0: estacionaria en tendencia)", kt[0], kt[1]))
    ols = sm.OLS(y, sm.add_constant(t)).fit()
    f.append(("Tendencia lineal (t de la pendiente)", ols.tvalues[1], ols.pvalues[1]))
    for lag in (6, 12):
        lb = acorr_ljungbox(y, lags=[lag], return_df=True)
        f.append((f"Ljung–Box rezago {lag}", lb["lb_stat"].iat[0], lb["lb_pvalue"].iat[0]))
    stl = STL(y.values, period=12, robust=True).fit()
    f.append(("Fuerza estacional STL (0–1)", max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.seasonal)), np.nan))
    f.append(("Fuerza de tendencia STL (0–1)", max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.trend)), np.nan))
    b = bds(AutoReg(y.values, lags=1, trend="ct").fit().resid, max_dim=2)
    f.append(("BDS dim. 2 sobre residuos AR(1) (H0: lineal)", float(np.atleast_1d(b[0])[0]), float(np.atleast_1d(b[1])[0])))
    return pd.DataFrame(f, columns=["prueba", "estadistico", "p_valor"]).assign(serie=nombre)


total_mes = panel.groupby("mes")["ventas_usd"].sum()
activas_mes = panel.groupby("mes")["activo"].mean()
pruebas = pd.concat([pruebas_serie(total_mes.values, "venta neta total"),
                     pruebas_serie(activas_mes.values, "proporción de relaciones con compra")])
pruebas.round(4).to_csv(DIR_TAB / "R02_pruebas_series.csv", index=False, encoding="utf-8-sig")
pruebas.round(4)

### Figura 2. Evolución de la venta y de las relaciones con compra
### Figura 3. Distribución, perfil comercial, concentración y estacionalidad

In [ ]:
def figura_evolucion():
    base = total_mes[total_mes.index.year == 2023].mean()
    x = total_mes.index.to_timestamp()
    fig, ax = plt.subplots(1, 2, figsize=(ANCHO, 5.2 * CM))
    ax[0].plot(x, 100 * total_mes / base, color=PALETA[0])
    ax[0].set_ylabel("Venta neta (promedio 2023 = 100)")
    ax[1].plot(x, 100 * activas_mes, color=PALETA[0])
    ax[1].set_ylabel("Relaciones con compra (%)")
    for a, et in zip(ax, ["(a)", "(b)"]):
        for f in (FIN_ENTRENAMIENTO, FIN_VALIDACION):
            a.axvline(pd.Period(f, "M").to_timestamp(), color=GRISES[1], lw=0.8, ls=":")
        a.xaxis.set_major_locator(mdates.YearLocator())
        a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        etiqueta(a, et)
    fig.tight_layout(w_pad=2)
    indices = {int(k): float(v) for k, v in (100 * total_mes.groupby(total_mes.index.year).mean() / base).items()}
    act_anual = {int(k): float(100 * v) for k, v in activas_mes.groupby(activas_mes.index.year).mean().items()}
    guardar_json({"indice_anual": indices, "pct_con_compra_anual": act_anual}, DIR_TAB / "R03_evolucion.json")
    return guardar(fig, "IV_fig2_evolucion")


def figura_distribucion():
    p = panel.join(atributos[["tratamiento"]], on="par_id")
    a = p[p.activo == 1]
    fig, ax = plt.subplots(2, 2, figsize=(ANCHO, 8.8 * CM))
    v = np.log10(a.ventas_usd.abs())
    bins = np.linspace(np.floor(v.min()), np.ceil(v.max()), 40)
    ax[0, 0].hist(np.log10(a.loc[a.ventas_usd > 0, "ventas_usd"]), bins=bins, color=PALETA[0], alpha=0.85,
                  label=f"Positiva ({coma(100 * (a.ventas_usd > 0).mean())} %)")
    ax[0, 0].hist(np.log10(-a.loc[a.ventas_usd < 0, "ventas_usd"]), bins=bins, color=PALETA[1], alpha=0.85,
                  label=f"Negativa ({coma(100 * (a.ventas_usd < 0).mean())} %)")
    ax[0, 0].set_xlabel("log$_{10}$ |venta neta mensual| (USD)")
    ax[0, 0].set_ylabel("Meses con movimiento")
    ax[0, 0].set_ylim(0, ax[0, 0].get_ylim()[1] * 1.45)
    ax[0, 0].legend(loc="upper left", fontsize=7, handlelength=1.2)
    orden = a.groupby("tratamiento")["ventas_usd"].apply(lambda s: s.abs().median()).sort_values().index
    bp = ax[0, 1].boxplot([np.log10(a.loc[a.tratamiento == t, "ventas_usd"].abs()) for t in orden], vert=False,
                          tick_labels=list(orden), patch_artist=True, widths=0.55,
                          flierprops={"markersize": 1, "alpha": 0.25, "markeredgecolor": GRISES[1]},
                          medianprops={"color": PALETA[1], "lw": 1.6})
    for b in bp["boxes"]:
        b.set(facecolor=GRIS_CLARO, edgecolor=AZUL_MARINO, lw=0.6)
    ax[0, 1].set_xlabel("log$_{10}$ |venta neta mensual| (USD)")
    tot = p.groupby("par_id")["ventas_usd"].apply(lambda s: s.abs().sum()).sort_values(ascending=False)
    cum = 100 * tot.cumsum() / tot.sum()
    xr = 100 * np.arange(1, len(tot) + 1) / len(tot)
    k20 = float(np.interp(20, xr, cum.values))
    ax[1, 0].plot(xr, cum.values, color=PALETA[0])
    ax[1, 0].plot([0, 100], [0, 100], color=GRISES[2], lw=0.8, ls=":")
    ax[1, 0].scatter([20], [k20], color=PALETA[1], s=16, zorder=3)
    ax[1, 0].annotate(f"20 % de las relaciones\n= {coma(k20, 0)} % de la venta", (20, k20), xytext=(32, 40),
                      fontsize=7.5, color=GRISES[0], arrowprops={"arrowstyle": "-", "color": GRISES[1], "lw": 0.6})
    ax[1, 0].set_xlabel("Relaciones (% acumulado)")
    ax[1, 0].set_ylabel("Venta acumulada (%)")
    c = p[(p.mes.dt.year >= 2023) & (p.mes.dt.year <= 2025)]
    pv = c.groupby([c.mes.dt.year, c.mes.dt.month])["ventas_usd"].sum().unstack()
    pv = 100 * pv.div(pv.sum(axis=1), axis=0)
    im = ax[1, 1].imshow(pv.values, cmap=AZULES, aspect="auto")
    ax[1, 1].set_xticks(range(12), [m[0] for m in MES_ES])
    ax[1, 1].set_yticks(range(len(pv)), pv.index)
    ax[1, 1].grid(False)
    for (i, j), val in np.ndenumerate(pv.values):
        ax[1, 1].text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6.8,
                      color="white" if val > pv.values.mean() * 1.15 else TINTA)
    cb = fig.colorbar(im, ax=ax[1, 1], fraction=0.05, pad=0.02)
    cb.set_label("% de la venta anual")
    for a_, et in zip(ax.ravel(), ["(a)", "(b)", "(c)", "(d)"]):
        etiqueta(a_, et)
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    med = a.assign(z=a.ventas_usd.abs()).groupby("tratamiento").z.median().round(0).to_dict()
    guardar_json({"pct_venta_20pct_relaciones": k20, "mediana_por_tipo_cliente": med,
                  "participacion_mensual_min_max": [float(pv.values.min()), float(pv.values.max())]},
                 DIR_TAB / "R04_distribucion.json")
    return guardar(fig, "IV_fig3_distribucion")


figura_evolucion()
figura_distribucion()

## Paso 3. Conjunto de pronóstico directo multihorizonte
Para cada relación *i*, mes de origen *t* y horizonte *h* = 1 a 6 se crea un caso con predictores calculados **solo con meses ≤ t**, el calendario del mes pronosticado y las series externas publicadas al cierre de *t*. Un único modelo usa *h* como predictor (estrategia directa agrupada). Aquí también se calculan los métodos de referencia: ingenuo, medias móviles, ingenuo estacional, Croston–SBA y el método de la empresa.

In [ ]:
def calendario(meses):
    import holidays
    fer = holidays.country_holidays("EC", years=range(2022, 2028))
    filas = []
    for m in meses:
        dias = pd.date_range(m.start_time, m.end_time, freq="D")
        f_hab = sum(1 for d in dias if d in fer and d.weekday() < 5)
        filas.append({"mes": m, "feriados_habiles": f_hab, "dias_habiles": int(sum(d.weekday() < 5 for d in dias)) - f_hab,
                      "iva_15": int(m >= pd.Period("2024-04", "M"))})     # IVA general del 15 % desde abril de 2024
    return pd.DataFrame(filas).set_index("mes")


def croston_sba(fila, alpha=0.1):
    """Croston con corrección de Syntetos–Boylan; se actualiza solo en los meses con compra."""
    out = np.full(len(fila), np.nan)
    z = p = None
    q = 1
    for t, y in enumerate(fila):
        if np.isnan(y):
            continue
        if y > 0:
            if z is None:
                z, p = y, q
            else:
                z, p = z + alpha * (y - z), p + alpha * (q - p)
            q = 1
        else:
            q += 1
        if z is not None:
            out[t] = (1 - alpha / 2) * z / p
    return out


ATRIB = ["tratamiento", "detalle_neg", "provincia", "region", "linea"]
REFERENCIAS = {"bl_naive": "Ingenuo (último mes)", "bl_media3": "Media móvil 3", "bl_media6": "Media móvil 6",
               "bl_empresa": EMP, "bl_empresa_oraculo": EMP_OR, "bl_estacional": "Ingenuo estacional",
               "bl_croston": CRO}


def construir(panel_, exog_, atr):
    meses = pd.period_range(PERIODO_INICIO, panel_["mes"].max(), freq="M")
    T = len(meses)
    pares = panel_.groupby("par_id").agg(cliente=("cliente", "first"), sku=("sku", "first"),
                                         primer_mes=("primer_mes", "first")).join(atr)
    P = len(pares)
    pos = {m: k for k, m in enumerate(meses)}
    fila = pd.Series(np.arange(P), index=pares.index)

    def ancho(col):
        M = np.zeros((P, T))
        M[fila[panel_["par_id"]].values, panel_["mes"].map(pos).values] = panel_[col].values
        return M
    Y, U = ancho("ventas_usd"), ancho("unidades")
    entrada = pares["primer_mes"].map(pos).values
    existe = np.arange(T)[None, :] >= entrada[:, None]

    def agregado(clave):
        g = pd.factorize(pares[clave])[0]
        A = np.zeros((g.max() + 1, T))
        np.add.at(A, g, Y)
        return A[g]
    YC, YS = agregado("cliente"), agregado("sku")

    def roll(M, k, fn):
        out = np.full_like(M, np.nan)
        for t in range(T):
            out[:, t] = fn(M[:, max(0, t - k + 1):t + 1], axis=1)
        return out
    actv = (Y != 0).astype(float)
    m3, m6, m12 = roll(Y, 3, np.mean), roll(Y, 6, np.mean), roll(Y, 12, np.mean)
    s6, mx6 = roll(Y, 6, np.std), roll(Y, 6, np.max)
    a6, a12 = roll(actv, 6, np.sum), roll(actv, 12, np.sum)
    c3, k3 = roll(YC, 3, np.mean), roll(YS, 3, np.mean)
    rec = np.full((P, T), np.nan)
    ult = np.full(P, -1)
    cum_pos, cum_y = np.cumsum(actv, 1), np.cumsum(Y, 1)
    for t in range(T):
        ult = np.where(actv[:, t] > 0, t, ult)
        rec[:, t] = np.where(ult >= 0, t - ult, np.nan)
    edad = np.arange(T)[None, :] - entrada[:, None] + 1
    tam_medio = np.where(cum_pos > 0, cum_y / np.maximum(cum_pos, 1), np.nan)
    adi = np.where(cum_pos > 0, edad / np.maximum(cum_pos, 1), np.nan)
    sba = np.vstack([croston_sba(np.where(existe[i], Y[i], np.nan)) for i in range(P)])
    # Método de la empresa: venta de la línea repartida según el promedio POSITIVO de 6 meses de cada cliente
    prom6 = roll(Y, 6, np.mean)
    A = np.where(prom6 > 0, prom6, 0)
    g_l = pd.factorize(pares["sku"])[0]
    D = np.zeros((g_l.max() + 1, T))
    np.add.at(D, g_l, A)
    D = D[g_l]
    peso_emp = np.where((prom6 > 0) & (D > 0), prom6 / np.where(D > 0, D, 1), 0.0)
    linea_prom6 = roll(YS, 6, np.mean)
    cal = calendario(meses)
    t_min = pos[pd.Period(PERIODO_INICIO, "M") + (MESES_HISTORIA_MIN - 1)]
    bloques = []
    for t in range(t_min, T - 1):
        idx = np.where(existe[:, t])[0]
        base = {"par_id": pares.index.values[idx], "origen": meses[t],
                **{f"lag_{k}": Y[idx, t - k] for k in range(6)},
                "media_3": m3[idx, t], "media_6": m6[idx, t], "media_12": m12[idx, t],
                "std_6": s6[idx, t], "max_6": mx6[idx, t], "activos_6": a6[idx, t], "activos_12": a12[idx, t],
                "recencia": rec[idx, t], "edad_relacion": edad[idx, t], "tam_medio_pos": tam_medio[idx, t],
                "adi_exp": adi[idx, t], "croston_sba": sba[idx, t],
                "cliente_media_3": c3[idx, t], "sku_media_3": k3[idx, t],
                "und_lag_0": U[idx, t], "und_media_6": U[idx, max(0, t - 5):t + 1].mean(1),
                "bl_naive": Y[idx, t], "bl_media3": m3[idx, t], "bl_media6": m6[idx, t],
                "peso_empresa": peso_emp[idx, t], "bl_empresa": linea_prom6[idx, t] * peso_emp[idx, t],
                "bl_croston": np.nan_to_num(sba[idx, t])}
        for c in ATRIB:
            base[c] = pares[c].values[idx]
        b = pd.DataFrame(base)
        for v in exog_.columns:
            b[f"x_{v}"] = exog_.loc[meses[t], v] if meses[t] in exog_.index else np.nan
        for h in HORIZONTES:
            if t + h >= T:
                continue
            bh = b.copy()
            bh["h"] = h
            bh["mes_objetivo"] = meses[t + h]
            bh["mes_cal_obj"] = meses[t + h].month
            for c in cal.columns:
                bh[f"{c}_obj"] = cal.loc[meses[t + h], c]
            j = t + h - 12
            bh["y_mismo_mes_anio_ant"] = Y[idx, j] if j >= 0 else np.nan
            bh["bl_estacional"] = np.nan_to_num(bh["y_mismo_mes_anio_ant"].values, nan=0.0) if j >= 0 \
                else bh["bl_naive"].values
            bh["bl_empresa_oraculo"] = YS[idx, t + h] * peso_emp[idx, t]
            bh["y"] = Y[idx, t + h]
            bloques.append(bh)
    df = pd.concat(bloques, ignore_index=True)
    fe, fv = pd.Period(FIN_ENTRENAMIENTO, "M"), pd.Period(FIN_VALIDACION, "M")
    df["particion"] = np.where(df["mes_objetivo"] <= fe, "entrenamiento",
                               np.where(df["mes_objetivo"] <= fv, "validacion", "prueba"))
    for c in ATRIB:                      # categorías: menos memoria, mismos valores
        df[c] = df[c].astype("category")
    return df


def columnas_predictoras(df, exog_usar):
    excl = {"par_id", "origen", "mes_objetivo", "y", "particion", "peso_empresa"} | set(REFERENCIAS)
    cols = [c for c in df.columns if c not in excl and not c.startswith("x_")]
    return cols + [f"x_{v}" for v in exog_usar]


t0 = time.perf_counter()
datos = construir(panel, exog, atributos)
print(f"Conjunto de pronóstico: {len(datos):,} casos en {time.perf_counter() - t0:.0f} s")
casos = datos.particion.value_counts()
print((100 * casos / casos.sum()).round(1).to_string())
guardar_json({"casos": len(datos), "por_particion": casos.to_dict(),
              "predictores_sin_exogenas": len(columnas_predictoras(datos, []))}, DIR_TAB / "R05_conjunto_pronostico.json")

## Paso 4. Correlación de Spearman y Figura 4
Casos de entrenamiento a un mes (*h* = 1). Panel (a): dispersión ADI–CV² con los cortes de Syntetos y Boylan. Panel (b): matriz de correlación de Spearman entre la venta del mes siguiente y los principales predictores.

In [ ]:
VAR_CORR = {"y": "Venta mes siguiente", "lag_0": "Venta mes de origen", "media_12": "Media 12 meses",
            "activos_6": "Meses con compra (6 m)", "adi_exp": "ADI", "croston_sba": "Croston–SBA",
            "y_mismo_mes_anio_ant": "Mismo mes año anterior", "x_ipc_general_var_a": "Inflación",
            "x_riesgo_pais": "Riesgo país", "x_tasa_activa": "Tasa activa", "x_remesas_var_a": "Remesas"}


def figura_intermitencia_correlacion():
    h = panel[panel.mes <= pd.Period(FIN_ENTRENAMIENTO, "M")]
    gg = h.groupby("par_id")
    tam = h[h.activo == 1].assign(z=lambda d: d.ventas_usd.abs()).groupby("par_id")["z"]
    d = pd.DataFrame({"adi": gg.size() / gg["activo"].sum().replace(0, np.nan),
                      "cv2": (tam.std(ddof=0) / tam.mean()) ** 2}).dropna()
    e = datos[(datos.particion == "entrenamiento") & (datos.h == 1)]
    cm = e[list(VAR_CORR)].corr("spearman")
    cm.round(3).to_csv(DIR_TAB / "R06_matriz_spearman_h1.csv", encoding="utf-8-sig")
    fig = plt.figure(figsize=(ANCHO, 7.6 * CM))
    g = fig.add_gridspec(1, 2, width_ratios=[0.8, 1.35], wspace=0.55)
    a1, a2 = fig.add_subplot(g[0]), fig.add_subplot(g[1])
    a1.scatter(d.adi, d.cv2.clip(upper=6), s=2.5, alpha=0.3, color=PALETA[0], edgecolor="none")
    a1.axvline(1.32, color=GRISES[0], lw=0.8)
    a1.axhline(0.49, color=GRISES[0], lw=0.8)
    a1.set_xscale("log")
    a1.set_ylim(-0.8, 6.9)
    a1.set_xlabel("ADI (meses)")
    a1.set_ylabel("CV² del monto")
    for xx, yy, t in [(1.4, -0.45, "Suave"), (1.4, 6.35, "Errática"), (25, -0.45, "Intermitente"), (25, 6.35, "Irregular")]:
        a1.text(xx, yy, t, ha="right" if xx > 10 else "left", fontsize=7, color=GRISES[0],
                bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1})
    M = cm.values[1:, :-1]
    n = M.shape[0]
    im = a2.imshow(np.ma.array(M, mask=np.triu(np.ones_like(M, bool), 1)), cmap=DIVERGENTE, vmin=-1, vmax=1)
    lab = list(VAR_CORR.values())
    a2.set_xticks(range(n), lab[:-1], rotation=45, ha="right", rotation_mode="anchor", fontsize=6.6)
    a2.set_yticks(range(n), lab[1:], fontsize=6.6)
    a2.grid(False)
    for s in a2.spines.values():
        s.set_visible(False)
    a2.tick_params(length=0)
    for i in range(n):
        for j in range(i + 1):
            v = M[i, j]
            a2.text(j, i, "0" if abs(v) < 0.005 else coma(v, 2), ha="center", va="center", fontsize=4.9,
                    color="white" if abs(v) > 0.6 else TINTA)
    cb = fig.colorbar(im, ax=a2, fraction=0.045, pad=0.02)
    cb.ax.tick_params(labelsize=6.3)
    cb.set_label("ρ de Spearman", fontsize=7)
    etiqueta(a1, "(a)", y=1.03)
    etiqueta(a2, "(b)", y=1.03)
    return guardar(fig, "IV_fig4_intermitencia_correlacion"), cm


_, matriz_spearman = figura_intermitencia_correlacion()
matriz_spearman["y"].round(2).sort_values()

## Paso 5. Decisiones en validación: transformación de la venta, exógenas y ACP
Con LightGBM como modelo de referencia, y **sin tocar la prueba**, se compara: (1) entrenar con la venta transformada *g(y)* o en niveles, y (2) cinco formas de incluir el entorno macroeconómico. Las especificaciones marcadas como «solo validación» usan series que no estarían disponibles en la prueba y no pueden elegirse.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

CATEG = ["tratamiento", "detalle_neg", "provincia", "region", "linea"]


class Codificador:
    """Ordinal para árboles; estandarización, indicadores de faltante y one-hot para el MLP."""

    def __init__(self, cols):
        self.cols = cols
        self.cat = [c for c in cols if c in CATEG]
        self.num = [c for c in cols if c not in CATEG]

    def ajustar(self, X):
        self.niveles = {c: pd.Index(sorted(X[c].astype(str).unique())) for c in self.cat}
        self.mediana = X[self.num].median()
        Z = self._num_mlp(X)
        self.mu, self.sd = Z.mean(), Z.std().replace(0, 1)
        return self

    def arbol(self, X):
        Z = X[self.cols].copy()
        for c in self.cat:
            Z[c] = self.niveles[c].get_indexer(Z[c].astype(str))
        return Z

    def _num_mlp(self, X):
        Z = X[self.num].astype(float)
        falt = Z.isna().astype(float).add_suffix("_na")
        falt = falt[[c for c in falt.columns if c[:-3] in ("media_12", "y_mismo_mes_anio_ant", "recencia",
                                                            "tam_medio_pos", "adi_exp", "croston_sba")]]
        Z = Z.fillna(self.mediana)
        for c in Z.columns:
            if c.startswith(("lag_", "media_", "std_", "max_", "tam_", "cliente_", "sku_", "und_", "croston", "y_mismo")):
                Z[c] = np.sign(Z[c]) * np.log1p(Z[c].abs())
        return pd.concat([Z, falt], axis=1)

    def mlp(self, X):
        Z = (self._num_mlp(X) - self.mu) / self.sd
        oh = [pd.get_dummies(pd.Categorical(X[c].astype(str), categories=self.niveles[c]), prefix=c)
              .astype(float).set_index(X.index) for c in self.cat]
        return pd.concat([Z] + oh, axis=1).values.astype(np.float32)


def _tr(y, modo):
    y = np.asarray(y, float)
    return np.sign(y) * np.log1p(np.abs(y)) if modo == "slog" else y


def _inv(p, modo):
    p = np.asarray(p, float)
    return np.sign(p) * np.expm1(np.abs(p)) if modo == "slog" else p


def ajustar_modelo(nombre, params, Xtr, ytr, cod, Xva=None, yva=None, modo="nivel"):
    """Devuelve (modelo, iteraciones). Los boosting usan parada temprana si se entrega validación."""
    yt = _tr(ytr, modo)
    cod.techo = float(np.max(np.abs(yt)))
    if nombre == "Random Forest":
        from sklearn.ensemble import RandomForestRegressor
        m = RandomForestRegressor(n_estimators=params["n_estimators"], min_samples_leaf=params["min_samples_leaf"],
                                  max_features=params["max_features"], max_samples=params["max_samples"],
                                  n_jobs=-1, random_state=SEMILLA)
        m.fit(cod.arbol(Xtr), yt)
        return m, None
    if nombre == "MLP":
        from sklearn.neural_network import MLPRegressor
        m = MLPRegressor(hidden_layer_sizes=params["capas"], alpha=params["alpha"], learning_rate_init=1e-3,
                         batch_size=1024, max_iter=100, early_stopping=True, validation_fraction=0.1,
                         n_iter_no_change=5, random_state=SEMILLA)
        cod.escala_y = float(np.std(yt)) or 1.0
        m.fit(cod.mlp(Xtr), yt / cod.escala_y)
        return m, None
    if nombre == "XGBoost":
        import xgboost as xgb
        m = xgb.XGBRegressor(n_estimators=params.get("n_iter", 3000), learning_rate=0.05,
                             max_depth=params["max_depth"], min_child_weight=params["min_child_weight"],
                             subsample=0.8, colsample_bytree=0.8, tree_method="hist", n_jobs=2,
                             random_state=SEMILLA, early_stopping_rounds=100 if Xva is not None else None)
        kw = {"eval_set": [(cod.arbol(Xva), _tr(yva, modo))], "verbose": False} if Xva is not None else {}
        m.fit(cod.arbol(Xtr), yt, **kw)
        return m, (m.best_iteration + 1 if Xva is not None else None)
    if nombre == "LightGBM":
        import lightgbm as lgb
        m = lgb.LGBMRegressor(n_estimators=params.get("n_iter", 3000), learning_rate=0.05,
                              num_leaves=params["num_leaves"], min_child_samples=params["min_child_samples"],
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=2,
                              random_state=SEMILLA, verbose=-1)
        kw = {}
        if Xva is not None:
            kw = {"eval_set": [(cod.arbol(Xva), _tr(yva, modo))], "callbacks": [lgb.early_stopping(100, verbose=False)]}
        m.fit(cod.arbol(Xtr), yt, categorical_feature=list(cod.cat), **kw)
        return m, (m.best_iteration_ if Xva is not None else None)
    if nombre == "CatBoost":
        from catboost import CatBoostRegressor
        m = CatBoostRegressor(iterations=params.get("n_iter", 3000), learning_rate=0.05, depth=params["depth"],
                              l2_leaf_reg=params["l2"], loss_function="RMSE", random_seed=SEMILLA, thread_count=2,
                              verbose=False, od_type="Iter" if Xva is not None else None,
                              od_wait=100 if Xva is not None else None, allow_writing_files=False)
        kw = {}
        if Xva is not None:
            kw = {"eval_set": (_cat(Xva, cod), _tr(yva, modo)), "use_best_model": True}
        m.fit(_cat(Xtr, cod), yt, cat_features=cod.cat, **kw)
        return m, (m.get_best_iteration() + 1 if Xva is not None else None)
    raise ValueError(nombre)


def _cat(X, cod):
    Z = X[cod.cols].copy()
    for c in cod.cat:
        Z[c] = Z[c].astype(str)
    return Z


def predecir(nombre, m, X, cod, modo):
    """Pronóstico en USD; se acota al máximo |y| del entrenamiento (en la escala de ajuste)."""
    if nombre == "MLP":
        p = m.predict(cod.mlp(X)) * getattr(cod, "escala_y", 1.0)
    elif nombre == "CatBoost":
        p = m.predict(_cat(X, cod))
    else:
        p = m.predict(cod.arbol(X))
    techo = getattr(cod, "techo", np.inf)
    return _inv(np.clip(p, -techo, techo), modo)


def agregar_pca(df, variables, nombre):
    """ACP ajustado SOLO con los meses de origen del entrenamiento; componentes hasta el 80 % de la varianza."""
    z = exog[variables].loc[pd.Period(PERIODO_INICIO, "M"):]
    ent = z.loc[:pd.Period(FIN_ENTRENAMIENTO, "M")].dropna()
    sc = StandardScaler().fit(ent)
    pca = PCA().fit(sc.transform(ent))
    k = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.80) + 1)
    comp = pd.DataFrame(pca.transform(sc.transform(z.ffill().bfill()))[:, :k], index=z.index,
                        columns=[f"{nombre}_pc{i + 1}" for i in range(k)])
    for c in comp.columns:
        df[f"x_{c}"] = df["origen"].map(comp[c])
    info = {"variables": variables, "k": k, "varianza_explicada": pca.explained_variance_ratio_[:k].tolist(),
            "cargas": pd.DataFrame(pca.components_[:k], columns=variables, index=comp.columns).round(3).to_dict()}
    return df, list(comp.columns), info


entval = datos[datos.particion != "prueba"].copy()
info_pca = {}
entval, pc_adm, info_pca["admitidas"] = agregar_pca(entval, ADMITIDAS, "pca_adm")
entval, pc_tod, info_pca["todas"] = agregar_pca(entval, TODAS_MACRO, "pca_tod")
guardar_json(info_pca, DIR_TAB / "R07_acp.json")
tr, va = entval[entval.particion == "entrenamiento"], entval[entval.particion == "validacion"]
ESPECS = {"Macro admitidas": ADMITIDAS, "Sin macro (calendario)": [],
          "Macro completas [solo validación]": TODAS_MACRO,
          "ACP macro admitidas": pc_adm, "ACP macro completas [solo validación]": pc_tod}
abl = []


def correr_ablacion(nombre, vars_, modo):
    cols = columnas_predictoras(entval, vars_)
    cod = Codificador(cols).ajustar(tr)
    m, it = ajustar_modelo("LightGBM", {"num_leaves": 31, "min_child_samples": 20}, tr, tr.y.values, cod,
                           va, va.y.values, modo)
    met = metricas(va.y, predecir("LightGBM", m, va, cod, modo))
    abl.append({"especificacion": nombre, "transformacion": modo, "n_exogenas": len(vars_), "iteraciones": it, **met})
    print(f"  {nombre:40s} [{modo}] WAPE validación = {met['WAPE']:.4f}")


for modo in TRANSFORMACIONES:
    correr_ablacion("Macro admitidas", ADMITIDAS, modo)
# La transformación logarítmica pronostica cerca de la mediana: puede bajar el WAPE por relación pero subestimar
# el total. Solo se admiten transformaciones con sesgo del total de validación dentro de ±SESGO_MAX_PCT.
_cand = [r for r in abl if abs(r["sesgo_total_pct"]) <= SESGO_MAX_PCT] or abl
MODO_Y = min(_cand, key=lambda r: r["WAPE"])["transformacion"]
for r in abl:
    print(f"  {r['transformacion']:6s} WAPE = {r['WAPE']:.4f} | sesgo del total = {r['sesgo_total_pct']:+.1f} %"
          + ("  (descartada por sesgo)" if r not in _cand else ""))
for nombre, vars_ in ESPECS.items():
    if nombre != "Macro admitidas":
        correr_ablacion(nombre, vars_, MODO_Y)
abl = pd.DataFrame(abl)
abl.to_csv(DIR_TAB / "R08_ablacion_validacion.csv", index=False, encoding="utf-8-sig")
factibles = abl[(abl.transformacion == MODO_Y) & ~abl.especificacion.str.contains("solo validación", regex=False)]
ESPEC = factibles.sort_values("WAPE").especificacion.iat[0]
EXOG_VARS = ESPECS[ESPEC]
if ESPEC.startswith("ACP"):
    datos, _, _ = agregar_pca(datos, ADMITIDAS, "pca_adm")
COLS = columnas_predictoras(entval, EXOG_VARS)
del entval
gc.collect()
guardar_json({"transformacion": MODO_Y, "especificacion_exogenas": ESPEC, "variables": EXOG_VARS,
              "n_predictores": len(COLS)}, DIR_TAB / "R09_decisiones_validacion.json")
print(f"\nElegido en validación: transformación = {MODO_Y} | exógenas = {ESPEC} | {len(COLS)} predictores")
abl[["especificacion", "transformacion", "WAPE", "RMSE", "sesgo_total_pct"]].round(4)

## Paso 6. Búsqueda de hiperparámetros, entrenamiento final y prueba
Cada configuración de la rejilla se evalúa con el WAPE de **validación** (RF y MLP sobre una submuestra estratificada por horizonte; los boosting con parada temprana). La mejor se reentrena con entrenamiento + validación y se aplica **una sola vez** a la prueba (abril a septiembre de 2026).

In [ ]:
def submuestra(df, n):
    if len(df) <= n:
        return df
    return df.groupby("h", group_keys=False).apply(lambda g: g.sample(int(n * len(g) / len(df)), random_state=SEMILLA))


def buscar(nombre, registro):
    cod = Codificador(COLS).ajustar(tr)
    trs = submuestra(tr, MAX_FILAS_BUSQUEDA) if nombre in ("Random Forest", "MLP") else tr
    mejor = None
    claves = list(REJILLAS[nombre])
    for valores in itertools.product(*REJILLAS[nombre].values()):
        params = dict(zip(claves, valores))
        t0 = time.perf_counter()
        m, n_it = ajustar_modelo(nombre, params, trs, trs.y.values, cod, va, va.y.values, MODO_Y)
        met = metricas(va.y.values, predecir(nombre, m, va, cod, MODO_Y))
        registro.append({"modelo": nombre, **{k: str(v) for k, v in params.items()}, "n_iter": n_it,
                         "WAPE_val": met["WAPE"], "RMSE_val": met["RMSE"], "seg": round(time.perf_counter() - t0, 1)})
        print(f"    {nombre} {params} it={n_it} WAPE val={met['WAPE']:.4f}")
        if mejor is None or met["WAPE"] < mejor[0]:
            mejor = (met["WAPE"], params, n_it)
    params = dict(mejor[1])
    if mejor[2] is not None:
        params["n_iter"] = int(mejor[2] * 1.1)       # más datos en el reajuste: 10 % más de árboles
    return params


prueba = datos[datos.particion == "prueba"].copy()
trva = pd.concat([tr, va])
del datos                                 # ya no se necesita: libera memoria
gc.collect()
registro, tiempos, HIPER, MODELOS_AJUSTADOS = [], [], {}, {}
for nombre in MODELOS:
    print(f"> {nombre}")
    t0 = time.perf_counter()
    HIPER[nombre] = buscar(nombre, registro)
    t_b = time.perf_counter() - t0
    cod = Codificador(COLS).ajustar(trva)
    t0 = time.perf_counter()
    m, _ = ajustar_modelo(nombre, HIPER[nombre], trva, trva.y.values, cod, modo=MODO_Y)
    t_e = time.perf_counter() - t0
    prueba[f"p_{nombre}"] = predecir(nombre, m, prueba, cod, MODO_Y)
    MODELOS_AJUSTADOS[nombre] = (m, cod)
    with open(DIR_INT / f"modelo_{nombre.replace(' ', '_')}.pkl", "wb") as f:
        pickle.dump({"modelo": m, "codificador": cod, "modo": MODO_Y, "params": HIPER[nombre], "cols": COLS}, f)
    tiempos.append({"modelo": nombre, "seg_busqueda": round(t_b, 1), "seg_entrenamiento": round(t_e, 1),
                    "filas_entrenamiento": len(trva), "filas_prueba": len(prueba)})
    print(f"  WAPE prueba = {wape(prueba.y, prueba[f'p_{nombre}']):.4f} (entrenamiento final {t_e:.0f} s)")
busqueda = pd.DataFrame(registro)
busqueda.to_csv(DIR_TAB / "R10_busqueda_hiperparametros.csv", index=False, encoding="utf-8-sig")
guardar_json({k: {kk: str(vv) for kk, vv in v.items()} for k, v in HIPER.items()}, DIR_TAB / "R11_hiperparametros.json")
pd.DataFrame(tiempos).to_csv(DIR_TAB / "R12_tiempos.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(tiempos)

## Paso 7. Evaluación en la prueba
Métricas totales, por horizonte y por tipo de relación; WAPE al sumar los pronósticos por cliente, línea y empresa; e intervalos de confianza bootstrap de la diferencia de WAPE frente al método de la empresa, remuestreando relaciones completas (conglomerados) para respetar la dependencia entre meses de una misma relación.

In [ ]:
def evaluar(df, col, grupos=("h", "segmento_sb")):
    filas = [{"grupo": "Total", **metricas(df.y, df[col])}]
    for g in grupos:
        for k, d in df.groupby(g, observed=True):
            filas.append({"grupo": f"{g}={k}", **metricas(d.y, d[col])})
    return pd.DataFrame(filas)


def wape_agregado(df, col, nivel):
    claves = ["mes_objetivo", "h"] + ([nivel] if nivel != "empresa" else [])
    a = df.groupby(claves)[["y", col]].sum()
    return float(np.abs(a.y - a[col]).sum() / a.y.abs().sum())


def bootstrap_dif_wape(df, col_a, col_b, n=N_BOOTSTRAP):
    rng = np.random.default_rng(SEMILLA)
    ea = (df[col_a] - df.y).abs().groupby(df.par_id).sum().values
    eb = (df[col_b] - df.y).abs().groupby(df.par_id).sum().values
    ys = df.y.abs().groupby(df.par_id).sum().values
    difs = np.empty(n)
    for b in range(n):
        s = rng.integers(0, len(ys), len(ys))
        difs[b] = (ea[s].sum() - eb[s].sum()) / ys[s].sum()
    obs = (ea.sum() - eb.sum()) / ys.sum()
    return float(obs), float(np.percentile(difs, 2.5)), float(np.percentile(difs, 97.5))


for b, nom in REFERENCIAS.items():
    prueba[f"p_{nom}"] = prueba[b]
prueba = prueba.join(panel.groupby("par_id")[["cliente", "sku"]].first(), on="par_id")
prueba["segmento_sb"] = prueba.par_id.map(seg_par)
NOMBRES = MODELOS + list(REFERENCIAS.values())
ev = pd.concat([evaluar(prueba, f"p_{n}").assign(modelo=n) for n in NOMBRES], ignore_index=True)
ev.to_csv(DIR_TAB / "R13_metricas_prueba.csv", index=False, encoding="utf-8-sig")
niveles = pd.DataFrame([{"modelo": n, "WAPE_relacion": wape(prueba.y, prueba[f"p_{n}"]),
                         **{f"WAPE_{lv}": wape_agregado(prueba, f"p_{n}", lv) for lv in ["cliente", "sku", "empresa"]}}
                        for n in NOMBRES]).rename(columns={"WAPE_sku": "WAPE_linea"})
niveles.to_csv(DIR_TAB / "R14_wape_niveles.csv", index=False, encoding="utf-8-sig")
bs = []
for n in [x for x in NOMBRES if x != EMP]:
    d, lo, hi = bootstrap_dif_wape(prueba, f"p_{n}", f"p_{EMP}")
    bs.append({"modelo": n, "dif_WAPE": d, "IC95_inf": lo, "IC95_sup": hi, "significativa": (lo > 0) or (hi < 0)})
bs = pd.DataFrame(bs)
bs.to_csv(DIR_TAB / "R15_bootstrap_vs_empresa.csv", index=False, encoding="utf-8-sig")
prueba.to_pickle(DIR_INT / "predicciones_prueba.pkl")
TOTAL = ev[ev.grupo == "Total"].set_index("modelo")
TOTAL[["WAPE", "MAE", "RMSE", "R2", "MAPE_y_pos", "sesgo_total_pct"]].sort_values("WAPE").round(4)

In [ ]:
bs.round(4)

## Paso 8. Curvas de aprendizaje y error por partición
Cada modelo se reajusta con la configuración elegida (solo entrenamiento) para registrar su error en entrenamiento y validación por iteración, y comparar el WAPE en las tres particiones: una brecha grande entre entrenamiento y validación indicaría sobreajuste.

In [ ]:
def params_sin_iter(nombre):
    return {k: v for k, v in HIPER[nombre].items() if k != "n_iter"}


cod_tr = Codificador(COLS).ajustar(tr)
ytr_t, yva_t = _tr(tr.y.values, MODO_Y), _tr(va.y.values, MODO_Y)
curvas, filas_part, pred_tr, pred_va = {}, [], {}, {}
for nombre in MODELOS:
    p = params_sin_iter(nombre)
    m, n_it = ajustar_modelo(nombre, p, tr, tr.y.values, cod_tr, va, va.y.values, MODO_Y)
    pred_tr[nombre] = predecir(nombre, m, tr, cod_tr, MODO_Y)
    pred_va[nombre] = predecir(nombre, m, va, cod_tr, MODO_Y)
    if nombre == "LightGBM":
        import lightgbm as lgb
        m2 = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=p["num_leaves"],
                               min_child_samples=p["min_child_samples"], subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, n_jobs=2, random_state=SEMILLA, verbose=-1)
        m2.fit(cod_tr.arbol(tr), ytr_t, eval_set=[(cod_tr.arbol(tr), ytr_t), (cod_tr.arbol(va), yva_t)],
               eval_metric="rmse", eval_names=["ent", "val"], categorical_feature=list(cod_tr.cat))
        r = m2.evals_result_
        curvas[nombre] = (r["ent"]["rmse"], r["val"]["rmse"], n_it, "Árboles")
    elif nombre == "XGBoost":
        import xgboost as xgb
        m2 = xgb.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=p["max_depth"],
                              min_child_weight=p["min_child_weight"], subsample=0.8, colsample_bytree=0.8,
                              tree_method="hist", n_jobs=2, random_state=SEMILLA, eval_metric="rmse")
        m2.fit(cod_tr.arbol(tr), ytr_t, eval_set=[(cod_tr.arbol(tr), ytr_t), (cod_tr.arbol(va), yva_t)], verbose=False)
        r = m2.evals_result()
        curvas[nombre] = (r["validation_0"]["rmse"], r["validation_1"]["rmse"], n_it, "Árboles")
    elif nombre == "CatBoost":
        from catboost import CatBoostRegressor, Pool
        m2 = CatBoostRegressor(iterations=800, learning_rate=0.05, depth=p["depth"], l2_leaf_reg=p["l2"],
                               loss_function="RMSE", random_seed=SEMILLA, thread_count=2, verbose=False,
                               allow_writing_files=False)
        m2.fit(Pool(_cat(tr, cod_tr), ytr_t, cat_features=cod_tr.cat),
               eval_set=Pool(_cat(va, cod_tr), yva_t, cat_features=cod_tr.cat))
        r = m2.get_evals_result()
        curvas[nombre] = (r["learn"]["RMSE"], r["validation"]["RMSE"], n_it, "Árboles")
    elif nombre == "MLP":
        ent = np.sqrt(2 * np.asarray(m.loss_curve_)) * cod_tr.escala_y
        val = np.sqrt(np.clip(1 - np.asarray(m.validation_scores_), 0, None)) * np.std(ytr_t)
        curvas[nombre] = (ent, val, int(np.argmax(m.validation_scores_)) + 1, "Épocas")
    else:
        Xs = cod_tr.arbol(tr).values
        idx = np.random.default_rng(SEMILLA + 3).choice(len(Xs), min(20000, len(Xs)), replace=False)
        Xv = cod_tr.arbol(va).values
        pt = np.cumsum([t_.predict(Xs[idx]) for t_ in m.estimators_], axis=0)
        pv = np.cumsum([t_.predict(Xv) for t_ in m.estimators_], axis=0)
        k = np.arange(1, len(m.estimators_) + 1)[:, None]
        curvas[nombre] = (np.sqrt(((pt / k - ytr_t[idx]) ** 2).mean(1)), np.sqrt(((pv / k - yva_t) ** 2).mean(1)),
                          None, "Árboles")
    filas_part.append({"modelo": nombre, "WAPE_entrenamiento": wape(tr.y, pred_tr[nombre]),
                       "WAPE_validacion": wape(va.y, pred_va[nombre]), "WAPE_prueba": wape(prueba.y, prueba[f"p_{nombre}"]),
                       "R2_entrenamiento": metricas(tr.y, pred_tr[nombre])["R2"],
                       "R2_validacion": metricas(va.y, pred_va[nombre])["R2"],
                       "R2_prueba": metricas(prueba.y, prueba[f"p_{nombre}"])["R2"]})
    print(filas_part[-1])
filas_part.append({"modelo": EMP, "WAPE_entrenamiento": wape(tr.y, tr.bl_empresa), "WAPE_validacion": wape(va.y, va.bl_empresa),
                   "WAPE_prueba": wape(prueba.y, prueba[f"p_{EMP}"]), "R2_entrenamiento": metricas(tr.y, tr.bl_empresa)["R2"],
                   "R2_validacion": metricas(va.y, va.bl_empresa)["R2"], "R2_prueba": metricas(prueba.y, prueba[f"p_{EMP}"])["R2"]})
particiones = pd.DataFrame(filas_part)
particiones.to_csv(DIR_TAB / "R16_error_por_particion.csv", index=False, encoding="utf-8-sig")
with open(DIR_INT / "curvas.pkl", "wb") as f:
    pickle.dump(curvas, f)
# serie total a un mes (Random Forest y método de la empresa) en las tres particiones
_t1 = prueba[prueba.h == 1]
_s = pd.concat([tr[tr.h == 1].assign(p=pred_tr["Random Forest"][tr.h.values == 1]).groupby("mes_objetivo")[["y", "p", "bl_empresa"]].sum().assign(particion="entrenamiento"),
                va[va.h == 1].assign(p=pred_va["Random Forest"][va.h.values == 1]).groupby("mes_objetivo")[["y", "p", "bl_empresa"]].sum().assign(particion="validacion"),
                _t1.assign(p=_t1["p_Random Forest"], bl_empresa=_t1[f"p_{EMP}"]).groupby("mes_objetivo")[["y", "p", "bl_empresa"]].sum().assign(particion="prueba")])
_s.to_csv(DIR_TAB / "R25_serie_total_h1.csv", encoding="utf-8-sig")
particiones.round(4)

## Paso 9. Combinación Random Forest + Croston–SBA con intervalo del 80 %
Random Forest suma bien los totales y Croston–SBA es fuerte en relaciones esporádicas. Su combinación $\hat{y}=w\,\hat{y}^{RF}+(1-w)\,\hat{y}^{SBA}$ se decide **en la validación**: el peso *w* ∈ {0; 0,1; …; 1} se elige por tipo de relación (o uno global, si así el error es menor). El intervalo del 80 % usa los cuantiles 10 y 90 de los errores de validación, relativos al nivel del pronóstico, agrupados por tipo de relación y por decil del pronóstico (conformal por grupos).

In [ ]:
REJ_W = np.round(np.linspace(0, 1, 11), 1)
va_c = va.copy()
va_c["segmento_sb"] = va_c.par_id.map(seg_par)
va_c["p_rf"], va_c["p_cro"] = pred_va["Random Forest"], va_c.bl_croston
te = prueba.copy()
te["p_rf"], te["p_cro"] = te["p_Random Forest"], te[f"p_{CRO}"]
w_glob = min(REJ_W, key=lambda w: wape(va_c.y, w * va_c.p_rf + (1 - w) * va_c.p_cro))
w_seg = {s: min(REJ_W, key=lambda w: wape(d.y, w * d.p_rf + (1 - w) * d.p_cro)) for s, d in va_c.groupby("segmento_sb")}
wv = va_c.segmento_sb.map(w_seg).values
por_tipo = wape(va_c.y, wv * va_c.p_rf + (1 - wv) * va_c.p_cro) < wape(va_c.y, w_glob * va_c.p_rf + (1 - w_glob) * va_c.p_cro)


def peso(d):
    return d.segmento_sb.map(w_seg).fillna(w_glob).values if por_tipo else w_glob


for d in (va_c, te):
    d["p_comb"] = peso(d) * d.p_rf + (1 - peso(d)) * d.p_cro
# Intervalo por grupos (conformal de Mondrian): los errores de validación, relativos al nivel del pronóstico, se agrupan
# por tipo de relación y por decil del pronóstico; cada caso de la prueba recibe los cuantiles 10 y 90 de su grupo.
CORTES_INT = np.unique(np.quantile(va_c.p_comb, np.linspace(0, 1, 11)))
CORTES_INT[0], CORTES_INT[-1] = -np.inf, np.inf


def grupo_int(d):
    return d.segmento_sb.astype(str) + "|" + pd.cut(d.p_comb, CORTES_INT, labels=False, include_lowest=True).astype(str)


esc_va, esc_te = va_c.p_comb.abs() + 1, te.p_comb.abs() + 1
r = (va_c.y - va_c.p_comb) / esc_va
q = r.groupby(grupo_int(va_c)).quantile([0.10, 0.90]).unstack()
qg = r.quantile([0.10, 0.90])
g_te = grupo_int(te)
te["p_comb_lo"] = te.p_comb + g_te.map(q[0.10]).fillna(qg[0.10]).values * esc_te
te["p_comb_hi"] = te.p_comb + g_te.map(q[0.90]).fillna(qg[0.90]).values * esc_te
cobertura = float(((te.y >= te.p_comb_lo) & (te.y <= te.p_comb_hi)).mean())
prueba["p_Combinación RF + Croston"] = te.p_comb
filas = []
for nom, col in [("Combinación RF + Croston", "p_comb"), ("Random Forest", "p_rf"), (CRO, "p_cro"), (EMP, f"p_{EMP}")]:
    mt = metricas(te.y, te[col])
    fila = {"metodo": nom, "WAPE_prueba": mt["WAPE"], "MAE": mt["MAE"], "RMSE": mt["RMSE"], "R2": mt["R2"],
            "sesgo_total_pct": mt["sesgo_total_pct"], "WAPE_cliente": wape_agregado(te, col, "cliente"),
            "WAPE_linea": wape_agregado(te, col, "sku"), "WAPE_empresa": wape_agregado(te, col, "empresa")}
    if col != f"p_{EMP}":
        d_, lo, hi = bootstrap_dif_wape(te, col, f"p_{EMP}")
        fila.update({"dif_vs_empresa": d_, "IC95_inf": lo, "IC95_sup": hi})
    filas.append(fila)
comb = pd.DataFrame(filas)
comb["cobertura_80"] = [cobertura, np.nan, np.nan, np.nan]
comb.to_csv(DIR_TAB / "R17_combinacion.csv", index=False, encoding="utf-8-sig")
pesos = pd.DataFrame({"tipo": list(w_seg) + ["global"], "w_RF": list(w_seg.values()) + [w_glob],
                      "usado": "por tipo" if por_tipo else "global"})
pesos.to_csv(DIR_TAB / "R18_combinacion_pesos.csv", index=False, encoding="utf-8-sig")
por_tipo_tab = te.groupby("segmento_sb").apply(lambda d: pd.Series({
    "relaciones": d.par_id.nunique(), "Combinación": wape(d.y, d.p_comb), "Random Forest": wape(d.y, d.p_rf),
    "Croston–SBA": wape(d.y, d.p_cro), "Método empresa": wape(d.y, d[f"p_{EMP}"]),
    "cobertura_80": ((d.y >= d.p_comb_lo) & (d.y <= d.p_comb_hi)).mean()}))
por_tipo_tab.to_csv(DIR_TAB / "R19_combinacion_por_tipo.csv", encoding="utf-8-sig")
te.to_pickle(DIR_INT / "predicciones_combinacion.pkl")
print(pesos.to_string(index=False))
comb.round(4)

## Paso 10. Réplica del estudio de Supermercados Tía (Guerra Espinoza y Flores Andrade, 2023)
**R1.** Se replica el diseño del estudio: predictores del mismo mes (unidades, mes, región, provincia y tipo de cliente), solo meses con venta, partición aleatoria 80/20, Random Forest de 20 árboles y red 15–10, con sus métricas (MSE y R²).
**R2.** La misma configuración dentro de nuestro protocolo (predictores del mes de origen, partición temporal, seis horizontes), para compararla en igualdad de condiciones con el método de la empresa y la propuesta.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

# Liberar memoria: el Random Forest ajustado (unos 200 MB) se vuelve a leer de disco cuando se necesite
MODELOS_AJUSTADOS.pop("Random Forest", None)
gc.collect()


def modelo_ajustado(nombre):
    if nombre not in MODELOS_AJUSTADOS:
        with open(DIR_INT / f"modelo_{nombre.replace(' ', '_')}.pkl", "rb") as f:
            o = pickle.load(f)
        return o["modelo"], o["codificador"]
    return MODELOS_AJUSTADOS[nombre]

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

p_ = panel.join(atributos[["region", "provincia", "tratamiento"]], on="par_id")
a_ = p_[p_.activo == 1].copy()
a_["mes_num"] = a_.mes.dt.month
num, cat = ["unidades", "mes_num"], ["region", "provincia", "tratamiento"]
Xtr1, Xte1, ytr1, yte1 = train_test_split(a_[num + cat], a_.ventas_usd.values, test_size=0.2, random_state=SEMILLA)
oh = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)
r1_modelos = {
    "Random Forest (config. Tía)": make_pipeline(ColumnTransformer([("n", "passthrough", num), ("c", oh(), cat)]),
                                                 RandomForestRegressor(n_estimators=20, max_features="sqrt", n_jobs=-1,
                                                                       random_state=SEMILLA)),
    "Red neuronal 15–10 (config. Tía)": make_pipeline(ColumnTransformer([("n", StandardScaler(), num), ("c", oh(), cat)]),
                                                      MLPRegressor(hidden_layer_sizes=(15, 10), batch_size=25, max_iter=25,
                                                                   random_state=SEMILLA))}
r1 = []
for n, m in r1_modelos.items():
    m.fit(Xtr1, ytr1)
    r1.append({"diseno": "R1", "modelo": n, **metricas(yte1, m.predict(Xte1))})
cod_tv = Codificador(COLS).ajustar(trva)
yt_tv = _tr(trva.y.values, MODO_Y)
techo = float(np.abs(yt_tv).max())
rf_t = RandomForestRegressor(n_estimators=20, max_features="sqrt", n_jobs=-1, random_state=SEMILLA).fit(cod_tv.arbol(trva), yt_tv)
prueba["p_RF Tía"] = _inv(np.clip(rf_t.predict(cod_tv.arbol(prueba)), -techo, techo), MODO_Y)
del rf_t
gc.collect()
esc = float(np.std(yt_tv)) or 1.0
red_t = MLPRegressor(hidden_layer_sizes=(15, 10), batch_size=25, max_iter=25, random_state=SEMILLA).fit(cod_tv.mlp(trva), yt_tv / esc)
prueba["p_Red Tía"] = _inv(np.clip(red_t.predict(cod_tv.mlp(prueba)) * esc, -techo, techo), MODO_Y)
del red_t
gc.collect()
r2 = [{"diseno": "R2", "modelo": n, **metricas(prueba.y, prueba[c])} for n, c in
      [("Random Forest (config. Tía)", "p_RF Tía"), ("Red neuronal 15–10 (config. Tía)", "p_Red Tía"),
       (EMP, f"p_{EMP}"), ("Random Forest (propuesta)", "p_Random Forest"),
       ("Combinación RF + Croston (propuesta)", "p_Combinación RF + Croston")]]
tia = pd.DataFrame(r1 + r2)
tia.to_csv(DIR_TAB / "R20_benchmark_tia.csv", index=False, encoding="utf-8-sig")
tia[["diseno", "modelo", "MSE", "R2", "RMSE", "MAE", "WAPE"]].round(4)

## Paso 11. Interpretabilidad: SHAP, LIME y permutación
- **SHAP** (TreeExplainer) sobre el modelo de *boosting* con menor WAPE de validación (no se elige con la prueba).
- **LIME** sobre tres relaciones típicas de la prueba (suave, intermitente e irregular).
- **Permutación por grupos** para los cinco modelos con la misma muestra: cuánto sube el WAPE al desordenar cada grupo de variables.

In [ ]:
import shap

mejor_val = busqueda.groupby("modelo").WAPE_val.min()
BOOST = mejor_val[["XGBoost", "LightGBM", "CatBoost"]].idxmin()
m_b, cod_b = modelo_ajustado(BOOST)
muestra = prueba.sample(min(N_SHAP, len(prueba)), random_state=SEMILLA)
Z = _cat(muestra, cod_b) if BOOST == "CatBoost" else cod_b.arbol(muestra)
sv = shap.TreeExplainer(m_b).shap_values(Z)
pd.to_pickle({"sv": sv, "Z": Z, "cols": COLS, "modelo": BOOST}, DIR_INT / "shap_muestra.pkl")  # para el anexo
imp_shap = pd.Series(np.abs(sv).mean(0), index=COLS).sort_values(ascending=False)
imp_shap.rename("media_abs_shap").to_csv(DIR_TAB / "R21_shap_importancia.csv", encoding="utf-8-sig")
print("Modelo interpretado con SHAP:", BOOST, f"(escala {MODO_Y})")

from lime.lime_tabular import LimeTabularExplainer

base_lime = cod_b.arbol(prueba.sample(min(20000, len(prueba)), random_state=SEMILLA + 1)).astype(float)


def f_pred(A):
    D = pd.DataFrame(A, columns=COLS)
    if BOOST == "CatBoost":
        for c in cod_b.cat:
            D[c] = cod_b.niveles[c][np.clip(D[c].round().astype(int), 0, len(cod_b.niveles[c]) - 1)]
    else:
        for c in cod_b.cat:
            D[c] = D[c].round().astype(int)
    return _inv(m_b.predict(D), MODO_Y)


lime_ex = LimeTabularExplainer(base_lime.fillna(base_lime.median()).values, feature_names=COLS,
                               categorical_features=[COLS.index(c) for c in cod_b.cat], mode="regression",
                               random_state=SEMILLA, discretize_continuous=True)
lime_filas = []
for s in ["suave", "intermitente", "irregular (lumpy)"]:
    d = prueba[(prueba.segmento_sb == s) & (prueba.h == 1) & (prueba.y != 0)]
    if len(d):
        x = cod_b.arbol(d.sample(1, random_state=SEMILLA)).astype(float).fillna(base_lime.median()).values[0]
        for regla, pesov in lime_ex.explain_instance(x, f_pred, num_features=6).as_list():
            lime_filas.append({"caso": s, "regla": regla, "peso": pesov})
lime_tab = pd.DataFrame(lime_filas)
lime_tab.to_csv(DIR_TAB / "R22_lime.csv", index=False, encoding="utf-8-sig")

GRUPOS = {"Ventas recientes": [c for c in COLS if c.startswith(("lag_", "media_", "std_", "max_"))],
          "Frecuencia e intermitencia": ["activos_6", "activos_12", "recencia", "adi_exp", "croston_sba", "tam_medio_pos",
                                         "edad_relacion"],
          "Venta del cliente y de la línea": ["cliente_media_3", "sku_media_3"],
          "Unidades": [c for c in COLS if c.startswith("und_")],
          "Calendario y estacionalidad": ["mes_cal_obj", "y_mismo_mes_anio_ant", "dias_habiles_obj", "feriados_habiles_obj",
                                          "iva_15_obj"],
          "Horizonte": ["h"], "Atributos comerciales": [c for c in CATEG if c in COLS],
          "Macroeconómicas": [c for c in COLS if c.startswith("x_")]}
mp = prueba.sample(min(N_PERMUTACION, len(prueba)), random_state=SEMILLA + 2)
rng = np.random.default_rng(SEMILLA)
perm = []
for nombre in MODELOS:
    m, cod = modelo_ajustado(nombre)
    base_w = wape(mp.y, predecir(nombre, m, mp, cod, MODO_Y))
    for g, cs in GRUPOS.items():
        cs = [c for c in cs if c in mp]
        if not cs:
            continue
        Xp = mp.copy()
        orden = rng.permutation(len(Xp))
        for c in cs:
            Xp[c] = Xp[c].values[orden]
        perm.append({"modelo": nombre, "grupo": g, "aumento_WAPE_pp": 100 * (wape(mp.y, predecir(nombre, m, Xp, cod, MODO_Y)) - base_w)})
perm = pd.DataFrame(perm)
perm.to_csv(DIR_TAB / "R23_permutacion.csv", index=False, encoding="utf-8-sig")
perm.pivot(index="grupo", columns="modelo", values="aumento_WAPE_pp")[MODELOS].round(2)

## Paso 12. Figuras del capítulo V
Cada figura responde a una pregunta del capítulo: ¿la base sintética es fiel?, ¿qué se decidió en la validación?, ¿los modelos aprenden sin sobreajustar?, ¿cuál pronostica mejor y con qué sesgo?, ¿cómo cambia el error con el horizonte, el tipo de relación y el nivel de agregación?, ¿la diferencia frente a la empresa es significativa?, ¿cómo se compara con la literatura?, ¿qué variables explican el pronóstico?

In [ ]:
# Todo se lee de disco: este paso puede ejecutarse solo (Paso 0 + Paso 1 + Paso 12) para rehacer las figuras
abl = pd.read_csv(DIR_TAB / "R08_ablacion_validacion.csv")
_dec = json.load(open(DIR_TAB / "R09_decisiones_validacion.json", encoding="utf-8"))
MODO_Y, ESPEC = _dec["transformacion"], _dec["especificacion_exogenas"]
busqueda = pd.read_csv(DIR_TAB / "R10_busqueda_hiperparametros.csv")
ev = pd.read_csv(DIR_TAB / "R13_metricas_prueba.csv")
TOTAL = ev[ev.grupo == "Total"].set_index("modelo")
niveles = pd.read_csv(DIR_TAB / "R14_wape_niveles.csv")
bs = pd.read_csv(DIR_TAB / "R15_bootstrap_vs_empresa.csv")
particiones = pd.read_csv(DIR_TAB / "R16_error_por_particion.csv")
comb = pd.read_csv(DIR_TAB / "R17_combinacion.csv")
cobertura = float(comb.cobertura_80.iat[0])
por_tipo_tab = pd.read_csv(DIR_TAB / "R19_combinacion_por_tipo.csv", index_col=0)
tia = pd.read_csv(DIR_TAB / "R20_benchmark_tia.csv")
imp_shap = pd.read_csv(DIR_TAB / "R21_shap_importancia.csv", index_col=0).iloc[:, 0]
lime_tab = pd.read_csv(DIR_TAB / "R22_lime.csv")
perm = pd.read_csv(DIR_TAB / "R23_permutacion.csv")
serie_h1 = pd.read_csv(DIR_TAB / "R25_serie_total_h1.csv")
curvas = pickle.load(open(DIR_INT / "curvas.pkl", "rb"))
te = pd.read_pickle(DIR_INT / "predicciones_combinacion.pkl")
BOOST = busqueda[busqueda.modelo.isin(["XGBoost", "LightGBM", "CatBoost"])].groupby("modelo").WAPE_val.min().idxmin()
NOM_VAR = {"adi_exp": "ADI", "tam_medio_pos": "Tamaño medio del pedido", "media_12": "Media 12 meses",
           "activos_12": "Meses con compra (12 m)", "activos_6": "Meses con compra (6 m)", "croston_sba": "Croston–SBA",
           "tratamiento": "Tipo de cliente", "media_3": "Media 3 meses", "media_6": "Media 6 meses",
           "lag_0": "Venta del mes de origen", "lag_1": "Venta hace 1 mes", "lag_2": "Venta hace 2 meses",
           "lag_3": "Venta hace 3 meses", "lag_4": "Venta hace 4 meses", "lag_5": "Venta hace 5 meses",
           "edad_relacion": "Antigüedad de la relación", "linea": "Línea", "y_mismo_mes_anio_ant": "Mismo mes año anterior",
           "und_media_6": "Unidades, media 6 m", "und_lag_0": "Unidades del mes de origen", "detalle_neg": "Detalle del negocio",
           "dias_habiles_obj": "Días hábiles", "recencia": "Meses desde la última compra",
           "cliente_media_3": "Venta del cliente (3 m)", "sku_media_3": "Venta de la línea (3 m)",
           "std_6": "Desviación 6 meses", "max_6": "Máximo 6 meses", "h": "Horizonte", "mes_cal_obj": "Mes pronosticado",
           "provincia": "Provincia", "region": "Región", "feriados_habiles_obj": "Feriados hábiles", "iva_15_obj": "IVA 15 %"}
for k in range(1, 6):
    NOM_VAR[f"x_pca_adm_pc{k}"] = f"Componente macro {k}"

NOM_CORTO = {EMP_OR: "Empresa (línea real)", "Ingenuo (último mes)": "Ingenuo", "Combinación RF + Croston": "Combinación RF + Croston"}


def color(n):
    if n in COLOR_MODELO:
        return COLOR_MODELO[n]
    if n.startswith("Comb"):
        return COLOR_COMB
    return REF_ESTILO.get(n, (GRISES[1], "-"))[0]


ETQ_VALIDACION = {
    "KS log|venta neta| (meses con movimiento)": "KS · monto de la venta", "TVD signo de la venta neta (positiva/negativa)": "TVD · signo de la venta",
    "KS log|unidades netas|": "KS · unidades", "KS log(precio implícito)": "KS · precio implícito",
    "KS tasa_actividad_par": "KS · tasa de actividad por relación", "KS compras_por_par": "KS · compras por relación",
    "KS ventas_totales_par_log": "KS · venta total por relación", "KS intervalo_entre_compras": "KS · intervalo entre compras",
    "KS recencia_final": "KS · recencia final", "KS pares por cliente": "KS · relaciones por cliente",
    "TVD mes calendario de meses activos": "TVD · mes de las compras", "TVD segmento de frecuencia (filas activas)": "TVD · segmento de frecuencia",
    "|Δ tasa de actividad| (pp)": "|Δ| tasa de actividad (pp)", "|Δ % meses sin pedido| (pp)": "|Δ| meses sin compra (pp)",
    "WAPE ventas mensuales agregadas": "WAPE · venta mensual agregada", "TVD perfil estacional de ventas": "TVD · perfil estacional",
    "Máx |Δρ Spearman| entre rasgos": "Máx. |Δρ| de Spearman", "Prop. sintéticos con DCR < P5(real holdout)": "Privacidad · DCR < P5"}

def fig_validacion_sintetica():
    ruta = DIR_TABLAS_SEGURO / "C_validacion_sintetica_estudio.csv"
    if not ruta.exists():
        print("Sin la tabla del entorno seguro: se omite la figura de validación de la base sintética.")
        return None
    c = pd.read_csv(ruta).dropna(subset=["umbral"])
    c["ratio"] = c.valor / c.umbral
    c = c.sort_values("ratio")
    etiquetas = [ETQ_VALIDACION.get(m_, re.sub(r"\s*\(.*\)", "", m_).replace("|Δ ACF media lag", "|Δ| ACF rezago").rstrip("|"))
                 for m_ in c.metrica]
    fig, ax = plt.subplots(figsize=(ANCHO, 8.4 * CM))
    yy = np.arange(len(c))
    ok = c.cumple.astype(str).str.lower().eq("true").values
    ax.hlines(yy, 0, c.ratio, color=[PALETA[0] if o else PALETA[1] for o in ok], lw=1.2)
    ax.scatter(c.ratio, yy, color=[PALETA[0] if o else PALETA[1] for o in ok], s=14, zorder=3)
    ax.axvline(1, color=GRISES[0], lw=0.9, ls="--")
    ax.text(1.02, -0.6, "umbral", fontsize=7, color=GRISES[0], va="bottom")
    ax.set_yticks(yy, etiquetas, fontsize=6.6)
    ax.set_xlabel("Valor observado / umbral fijado antes de generar (≤ 1 cumple)")
    fig.tight_layout()
    guardar_json({"criterios": int(len(c)), "cumplidos": int(ok.sum())}, DIR_TAB / "R24_validacion_sintetica.json")
    return guardar(fig, "V_validacion_base_sintetica")


def fig_ablacion():
    ab = abl.copy()
    ab["etq"] = ab.especificacion.str.replace(" [solo validación]", "*", regex=False) + \
        np.where(ab.transformacion == "slog", " · g(y)", " · niveles")
    ab = ab.sort_values("WAPE", ascending=False)
    eleg = ~ab.especificacion.str.contains("solo validación", regex=False)
    mejor = ab[eleg & (ab.transformacion == MODO_Y)].WAPE.min()
    fig, ax = plt.subplots(figsize=(ANCHO, 5.0 * CM))
    cols = [PALETA[0] if (w == mejor and e) else (GRISES[2] if not e else GRISES[1]) for w, e in zip(ab.WAPE, eleg)]
    ax.barh(ab.etq, 100 * ab.WAPE, color=cols, height=0.62)
    for k, w in enumerate(ab.WAPE):
        ax.text(100 * w + 0.3, k, coma(100 * w), va="center", fontsize=7.3, color=GRISES[0])
    ax.set_xlim(100 * ab.WAPE.min() - 6, 100 * ab.WAPE.max() + 4)
    ax.set_xlabel("WAPE de validación (%) — LightGBM de referencia")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return guardar(fig, "V_ablacion_validacion")


def fig_aprendizaje():
    fig, ax = plt.subplots(2, 3, figsize=(ANCHO, 9.4 * CM))
    ax = ax.ravel()
    for k, (a, n) in enumerate(zip(ax, MODELOS)):
        et, evv, best, xl = curvas[n]
        esc_ = 1e3 if MODO_Y == "nivel" else 1
        a.plot(np.arange(1, len(et) + 1), np.asarray(et) / esc_, color=COLOR_MODELO[n], label="Entrenamiento")
        a.plot(np.arange(1, len(evv) + 1), np.asarray(evv) / esc_, color=GRISES[0], ls="--", label="Validación")
        if best:
            a.axvline(best, color=GRISES[1], lw=0.8, ls=":")
        a.set_xlabel(xl)
        if k % 3 == 0:
            a.set_ylabel("RMSE (miles USD)" if MODO_Y == "nivel" else "RMSE (escala g(y))")
        etiqueta(a, f"({'abcde'[k]}) {n}")
    h_, l_ = ax[0].get_legend_handles_labels()
    fig.legend(h_, l_, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.01), fontsize=7.5)
    a = ax[5]
    for k, n in enumerate(MODELOS):
        w = 100 * busqueda[busqueda.modelo == n].WAPE_val.values
        a.scatter(np.full(len(w), k) + np.linspace(-0.15, 0.15, len(w)), w, color=COLOR_MODELO[n], s=12)
        a.scatter([k], [w.min()], s=60, facecolors="none", edgecolors=TINTA, lw=0.9)
    a.set_xticks(range(len(MODELOS)), ["RF", "MLP", "XGB", "LGBM", "CatB"], rotation=45, ha="right", rotation_mode="anchor")
    a.set_ylabel("WAPE validación (%)")
    etiqueta(a, "(f) Búsqueda")
    fig.tight_layout(h_pad=1.6, w_pad=1.2, rect=(0, 0.05, 1, 1))
    return guardar(fig, "V_curvas_aprendizaje")


def fig_particion():
    d = particiones.set_index("modelo")
    orden = MODELOS + [EMP]
    fig, ax = plt.subplots(figsize=(ANCHO, 5.4 * CM))
    xx = np.arange(len(orden))
    for k, (col, et, c) in enumerate([("WAPE_entrenamiento", "Entrenamiento", "#A9BFD2"),
                                       ("WAPE_validacion", "Validación", AZUL), ("WAPE_prueba", "Prueba", AZUL_MARINO)]):
        vals = 100 * d.loc[orden, col].values
        ax.bar(xx + (k - 1) * 0.27, np.minimum(vals, 150), width=0.25, color=c, label=et)
        for x_, v in zip(xx, vals):
            if v > 150:
                ax.text(x_ + (k - 1) * 0.27, 148, f"{v:.0f}", ha="center", va="top", fontsize=6.5, color="white", rotation=90)
    ax.set_xticks(xx, ["Random\nForest", "MLP", "XGBoost", "LightGBM", "CatBoost", "Método\nempresa"])
    ax.set_ylabel("WAPE (%)")
    ax.set_ylim(0, 150)
    ax.legend(ncol=3, loc="upper left")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    return guardar(fig, "V_error_por_particion")


def fig_wape_sesgo():
    sel = MODELOS + ["Combinación RF + Croston", CRO, EMP, EMP_OR, "Media móvil 6", "Ingenuo (último mes)", "Ingenuo estacional"]
    met = TOTAL.reindex(sel).copy()
    met.loc["Combinación RF + Croston", ["WAPE", "sesgo_total_pct"]] = comb.set_index("metodo").loc[
        "Combinación RF + Croston", ["WAPE_prueba", "sesgo_total_pct"]].values
    met = met.sort_values("WAPE")
    fig, ax = plt.subplots(1, 2, figsize=(ANCHO, 7.0 * CM), sharey=True, gridspec_kw={"width_ratios": [1.15, 1]})
    yy = np.arange(len(met))[::-1]
    cs = [color(n) for n in met.index]
    lim_w = 160
    ax[0].barh(yy, np.minimum(100 * met.WAPE, lim_w), color=cs, height=0.64)
    for y_, w in zip(yy, met.WAPE):
        ax[0].text(min(100 * w, lim_w) + 1, y_, coma(100 * w) + (" →" if 100 * w > lim_w else ""), va="center",
                   fontsize=7, color=GRISES[0])
    ax[0].set_yticks(yy, [NOM_CORTO.get(n, n) for n in met.index])
    ax[0].set_xlim(0, lim_w + 25)
    ax[0].set_xlabel("WAPE por relación y mes (%)")
    ax[1].barh(yy, met.sesgo_total_pct.clip(-60, 60), color=cs, height=0.64)
    ax[1].axvline(0, color=GRISES[0], lw=0.8)
    for y_, b in zip(yy, met.sesgo_total_pct):
        bb = np.clip(b, -60, 60)
        ax[1].text(bb + (2 if b >= 0 else -2), y_, ("+" if b >= 0 else "−") + coma(abs(b)), va="center",
                   ha="left" if b >= 0 else "right", fontsize=7, color=GRISES[0])
    ax[1].set_xlim(-75, 75)
    ax[1].set_xlabel("Sesgo del total pronosticado (%)")
    for a in ax:
        a.grid(axis="y", visible=False)
    etiqueta(ax[0], "(a)")
    etiqueta(ax[1], "(b)")
    fig.tight_layout(w_pad=1)
    return guardar(fig, "V_wape_sesgo")


def fig_horizonte_tipo_nivel():
    hh = ev[ev.grupo.str.startswith("h=")].copy()
    hh["h"] = hh.grupo.str[2:].astype(int)
    fig = plt.figure(figsize=(ANCHO, 11.8 * CM))
    g = fig.add_gridspec(2, 2, height_ratios=[1, 1.05], hspace=0.55, wspace=0.35)
    a0 = fig.add_subplot(g[0, :])
    for n in MODELOS:
        d = hh[hh.modelo == n]
        a0.plot(d.h, 100 * d.WAPE, marker="o", ms=3.5, color=COLOR_MODELO[n], label=n)
    for n in [CRO, EMP]:
        d = hh[hh.modelo == n]
        c, ls = REF_ESTILO[n]
        a0.plot(d.h, 100 * d.WAPE, ls=ls, color=c, lw=1.2, label=n)
    a0.set_xlabel("Horizonte h (meses después del origen)")
    a0.set_ylabel("WAPE (%)")
    lo_, hi_ = a0.get_ylim()
    hi_ = min(hi_, 140)
    a0.set_ylim(lo_, hi_ + 0.45 * (hi_ - lo_))
    a0.legend(ncol=4, loc="upper left", fontsize=7)
    etiqueta(a0, "(a)")
    a1 = fig.add_subplot(g[1, 0])
    tt = por_tipo_tab.rename(index=NOM_SEG).reindex(["Suave", "Errática", "Intermitente", "Irregular", "Nueva"]).dropna(how="all")
    yy = np.arange(len(tt))[::-1]
    for k, (n, c, hatch) in enumerate([("Combinación", COLOR_COMB, None), ("Random Forest", COLOR_MODELO["Random Forest"], None),
                                       ("Método empresa", COLOR_EMP, None), ("Croston–SBA", GRISES[1], "///")]):
        a1.barh(yy + (1.5 - k) * 0.2, np.minimum(100 * tt[n].values, 200), height=0.19, color=c, hatch=hatch,
                edgecolor="white", lw=0.3, label=n)
    a1.set_yticks(yy, tt.index)
    a1.set_xlim(0, 200)
    a1.set_xlabel("WAPE (%)")
    a1.grid(axis="y", visible=False)
    etiqueta(a1, "(b) Tipo de relación")
    a2 = fig.add_subplot(g[1, 1])
    nv = niveles.set_index("modelo")
    cb_ = comb.set_index("metodo").loc["Combinación RF + Croston"]
    nv.loc["Combinación"] = [cb_.WAPE_prueba, cb_.WAPE_cliente, cb_.WAPE_linea, cb_.WAPE_empresa]
    cols_n = [("WAPE_relacion", "Relación"), ("WAPE_cliente", "Cliente"), ("WAPE_linea", "Línea"), ("WAPE_empresa", "Empresa")]
    xx = np.arange(4)
    for k, (n, c, hatch) in enumerate([("Combinación", COLOR_COMB, None), ("Random Forest", COLOR_MODELO["Random Forest"], None),
                                       (EMP, COLOR_EMP, None), (CRO, GRISES[1], "///")]):
        a2.bar(xx + (k - 1.5) * 0.2, 100 * nv.loc[n, [c_ for c_, _ in cols_n]].values.astype(float), width=0.19,
               color=c, hatch=hatch, edgecolor="white", lw=0.3)
    a2.set_xticks(xx, [e for _, e in cols_n])
    a2.set_ylabel("WAPE (%)")
    a2.grid(axis="x", visible=False)
    etiqueta(a2, "(c) Nivel de agregación")
    h_, l_ = a1.get_legend_handles_labels()
    fig.legend(h_, l_, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.02), fontsize=7)
    fig.subplots_adjust(bottom=0.14)
    return guardar(fig, "V_horizonte_tipo_nivel")


def fig_serie_total():
    s = serie_h1.copy()
    s["mes"] = pd.PeriodIndex(s.mes_objetivo, freq="M").to_timestamp()
    base = s[s.particion == "entrenamiento"].y.mean()
    fig, ax = plt.subplots(figsize=(ANCHO, 5.6 * CM))
    for prt, et, alto in [("entrenamiento", "Entrenamiento", 0.97), ("validacion", "Validación", 0.97), ("prueba", "Prueba", 0.88)]:
        d = s[s.particion == prt]
        ax.axvspan(d.mes.min() - pd.Timedelta(days=15), d.mes.max() + pd.Timedelta(days=15), color=FONDO_PART[prt], zorder=0)
        ax.text(d.mes.min(), alto, et, transform=ax.get_xaxis_transform(), fontsize=7, color=GRISES[0], va="top")
    ax.plot(s.mes, 100 * s.y / base, color=TINTA, marker="o", ms=2.5, label="Observado")
    ax.plot(s.mes, 100 * s.p / base, color=COLOR_MODELO["Random Forest"], marker="s", ms=2.5, label="Random Forest")
    ax.plot(s.mes, 100 * s.bl_empresa / base, color=COLOR_EMP, ls="--", lw=1.1, label="Método empresa")
    ax.set_ylabel("Venta neta total, h = 1\n(media de entrenamiento = 100)")
    ax.set_ylim(0, None)
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.13))
    fig.tight_layout()
    return guardar(fig, "V_serie_total")


def fig_bootstrap():
    b = bs.copy()
    c_ = comb.set_index("metodo").loc["Combinación RF + Croston"]
    b = pd.concat([b, pd.DataFrame([{"modelo": "Combinación RF + Croston", "dif_WAPE": c_.dif_vs_empresa,
                                     "IC95_inf": c_.IC95_inf, "IC95_sup": c_.IC95_sup}])])
    b = b[b.modelo.isin(MODELOS + ["Combinación RF + Croston", CRO, EMP_OR, "Media móvil 6", "Ingenuo estacional"])]
    b = b.sort_values("dif_WAPE", ascending=False)
    lim = 40
    fig, ax = plt.subplots(figsize=(ANCHO, 5.8 * CM))
    for y_, r_ in zip(range(len(b)), b.itertuples()):
        c = color(r_.modelo)
        sig = (r_.IC95_inf > 0) or (r_.IC95_sup < 0)
        lo, hi, dv = (np.clip(100 * v, -lim, lim) for v in (r_.IC95_inf, r_.IC95_sup, r_.dif_WAPE))
        ax.plot([lo, hi], [y_, y_], color=c, lw=1.2)
        ax.scatter(dv, y_, color=c if sig else "white", edgecolor=c, s=26, zorder=3, lw=1.2)
        if 100 * r_.dif_WAPE > lim:
            ax.text(lim, y_ + 0.25, f"{coma(100 * r_.dif_WAPE, 0)} →", fontsize=6.5, ha="right", color=GRISES[0])
    ax.axvline(0, color=GRISES[0], lw=0.8)
    ax.set_yticks(range(len(b)), [NOM_CORTO.get(n, n) for n in b.modelo])
    ax.set_xlim(-lim - 2, lim + 2)
    ax.set_xlabel("WAPE del método − WAPE del método de la empresa (pp); IC 95 % bootstrap por relación")
    ax.text(-lim, len(b) - 0.4, "← mejor que la empresa", fontsize=7, color=GRISES[0])
    ax.text(lim, len(b) - 0.4, "peor que la empresa →", fontsize=7, color=GRISES[0], ha="right")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return guardar(fig, "V_bootstrap")


def fig_tia():
    t = tia.copy()
    etq = {("R1", "Random Forest (config. Tía)"): "R1 · RF Tía", ("R1", "Red neuronal 15–10 (config. Tía)"): "R1 · Red Tía",
           ("R2", "Random Forest (config. Tía)"): "R2 · RF Tía", ("R2", "Red neuronal 15–10 (config. Tía)"): "R2 · Red Tía",
           ("R2", EMP): "R2 · Empresa", ("R2", "Random Forest (propuesta)"): "R2 · RF propuesta",
           ("R2", "Combinación RF + Croston (propuesta)"): "R2 · Combinación"}
    t["etq"] = [etq[(a, b)] for a, b in zip(t.diseno, t.modelo)]
    cs = [GRISES[2], GRISES[2], GRISES[1], GRISES[1], COLOR_EMP, COLOR_MODELO["Random Forest"], COLOR_COMB]
    fig, ax = plt.subplots(1, 2, figsize=(ANCHO, 6.4 * CM))
    xx = np.arange(len(t))
    ax[0].bar(xx, t.R2.clip(lower=0), color=cs, width=0.62)
    for x_, v in zip(xx, t.R2):
        ax[0].text(x_, max(v, 0) + 0.02, coma(v, 2), ha="center", fontsize=6.8, color=GRISES[0])
    ax[0].set_ylabel("R² (mayor es mejor)")
    ax[0].set_ylim(0, 1)
    ax[1].bar(xx, 100 * t.WAPE, color=cs, width=0.62)
    for x_, v in zip(xx, t.WAPE):
        ax[1].text(x_, 100 * v + 2, coma(100 * v), ha="center", fontsize=6.8, color=GRISES[0])
    ax[1].set_ylabel("WAPE % (menor es mejor)")
    for a, et in zip(ax, ["(a)", "(b)"]):
        a.set_xticks(xx, t.etq, rotation=55, ha="right", rotation_mode="anchor", fontsize=7)
        a.grid(axis="x", visible=False)
        etiqueta(a, et)
    fig.tight_layout(w_pad=2)
    return guardar(fig, "V_benchmark_tia")


def fig_pronosticos_relaciones():
    """Una relación por tipo, elegida con una regla fija (la de mayor venta histórica de su tipo, con al
    menos 12 meses con compra hasta marzo de 2026), sin mirar la prueba. Se muestran dos usos del modelo:
    el plan semestral hecho en marzo de 2026 (h = 1 a 6) y la actualización mensual (h = 1, origen móvil)."""
    fin = pd.Period(FIN_VALIDACION, "M")
    hist_all = panel[panel.mes <= fin]
    g = hist_all.groupby("par_id")
    est = pd.DataFrame({"vol": g.ventas_usd.apply(lambda v: v.abs().sum()), "compras": g.activo.sum()})
    est = est[est.index.isin(te[te.origen == fin].par_id.unique()) & (est.compras >= 12)]
    est["tipo"] = est.index.map(seg_par)
    elegidos = []
    for tipo in ["suave", "errática", "intermitente", "irregular (lumpy)"]:
        d = est[est.tipo == tipo]
        if len(d):
            elegidos.append(d.vol.idxmax())
    plan = te[te.origen == fin]
    movil = te[te.h == 1]
    fig, ax = plt.subplots(2, 2, figsize=(ANCHO, 10.6 * CM))
    for k, (a, pid) in enumerate(zip(ax.ravel(), elegidos)):
        hist = panel[(panel.par_id == pid) & (panel.mes >= pd.Period("2025-01", "M"))]
        f = plan[plan.par_id == pid].sort_values("mes_objetivo")
        m = movil[movil.par_id == pid].sort_values("mes_objetivo")
        xf, xm = f.mes_objetivo.dt.to_timestamp(), m.mes_objetivo.dt.to_timestamp()
        a.axvline(fin.to_timestamp(), color=GRISES[1], lw=0.8, ls=":")
        a.fill_between(xf, f.p_comb_lo / 1e3, f.p_comb_hi / 1e3, color=COLOR_BANDA, alpha=0.55, lw=0, label="Intervalo del 80 % (plan)")
        a.plot(hist.mes.dt.to_timestamp(), hist.ventas_usd / 1e3, color=TINTA, marker="o", ms=2.5, lw=1.2, label="Observado")
        a.plot(xf, f.p_comb / 1e3, color=COLOR_COMB, lw=1.3, label="Plan de marzo de 2026 (h = 1 a 6)")
        a.plot(xm, m.p_comb / 1e3, color=AZUL_MARINO, marker="D", ms=2.8, lw=1.0, label="Actualización mensual (h = 1)")
        a.plot(xf, f[f"p_{EMP}"] / 1e3, color=COLOR_EMP, ls="--", lw=1.1, label="Método empresa")
        a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
        a.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
        if k % 2 == 0:
            a.set_ylabel("Miles de USD")
        etiqueta(a, f"({'abcd'[k]}) {hist.cliente.iat[0]} · {hist.sku.iat[0]} · {NOM_SEG.get(seg_par[pid], '')}")
    h_, l_ = ax[0, 0].get_legend_handles_labels()
    orden = [l_.index(x) for x in ["Observado", "Plan de marzo de 2026 (h = 1 a 6)", "Intervalo del 80 % (plan)",
                                   "Actualización mensual (h = 1)", "Método empresa"]]
    fig.legend([h_[i] for i in orden], [l_[i] for i in orden], ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.01), fontsize=7)
    fig.tight_layout(rect=(0, 0.09, 1, 1), h_pad=1.8)
    pd.DataFrame({"par_id": elegidos, "tipo": [seg_par[p] for p in elegidos]}).to_csv(
        DIR_TAB / "R26_relaciones_ilustradas.csv", index=False)
    return guardar(fig, "V_pronosticos_cliente_linea")


def regla_legible(r):
    var = re.findall(r"[A-Za-z_][A-Za-z_0-9]*", r)[0]
    if "=" in r and not re.search(r"[<>]", r):
        return f"{NOM_VAR.get(var, var)} = categoría {r.split('=')[1].strip()}"
    r = r.replace(var, NOM_VAR.get(var, var))
    r = re.sub(r"-?\d+\.\d+", lambda m: (f"{float(m.group()):,.0f}".replace(",", " ") if abs(float(m.group())) >= 50
                                          else coma(float(m.group()), 2)), r)
    return r.replace("<=", "≤")


def fig_interpretabilidad():
    fig = plt.figure(figsize=(ANCHO, 12.6 * CM))
    g = fig.add_gridspec(2, 2, width_ratios=[1, 1.05], height_ratios=[1, 1], hspace=0.55, wspace=0.95)
    a0 = fig.add_subplot(g[:, 0])
    top = imp_shap.head(12)[::-1]
    a0.barh([NOM_VAR.get(k, k) for k in top.index], top.values / (1e3 if MODO_Y == "nivel" else 1), color=PALETA[0], height=0.62)
    a0.set_xlabel("Media |SHAP| (miles USD)" if MODO_Y == "nivel" else "Media |SHAP| (escala g(y))")
    a0.grid(axis="y", visible=False)
    etiqueta(a0, f"(a) SHAP · {BOOST}")
    a1 = fig.add_subplot(g[0, 1])
    pv = perm.pivot(index="grupo", columns="modelo", values="aumento_WAPE_pp")[MODELOS]
    pv = pv.loc[pv["Random Forest"].sort_values(ascending=False).index]
    lim = np.abs(pv.values).max()
    im = a1.imshow(pv.values, cmap=DIVERGENTE.reversed(), vmin=-lim, vmax=lim, aspect="auto")
    a1.set_xticks(range(5), ["RF", "MLP", "XGB", "LGBM", "CatB"], fontsize=7)
    a1.set_yticks(range(len(pv)), pv.index, fontsize=6.8)
    a1.grid(False)
    for (i, j), v in np.ndenumerate(pv.values):
        a1.text(j, i, coma(v), ha="center", va="center", fontsize=6, color="white" if abs(v) > 0.6 * lim else TINTA)
    cb = fig.colorbar(im, ax=a1, fraction=0.06, pad=0.03)
    cb.ax.tick_params(labelsize=6.3)
    cb.set_label("Aumento del WAPE (pp)", fontsize=6.8)
    etiqueta(a1, "(b) Permutación por grupos")
    a2 = fig.add_subplot(g[1, 1])
    caso = "irregular (lumpy)" if "irregular (lumpy)" in set(lime_tab.caso) else lime_tab.caso.iat[0]
    d = lime_tab[lime_tab.caso == caso].iloc[::-1]
    a2.barh([regla_legible(r) for r in d.regla], d.peso / (1e3 if MODO_Y == "nivel" else 1),
            color=[PALETA[2] if v > 0 else PALETA[1] for v in d.peso], height=0.6)
    a2.axvline(0, color=GRISES[0], lw=0.8)
    a2.tick_params(axis="y", labelsize=6.3)
    a2.set_xlabel("Efecto en el pronóstico (miles USD)" if MODO_Y == "nivel" else "Efecto (escala g(y))")
    a2.grid(axis="y", visible=False)
    etiqueta(a2, f"(c) LIME · relación {NOM_SEG.get(caso, caso).lower()}")
    return guardar(fig, "V_interpretabilidad")


figuras_v = [fig_validacion_sintetica(), fig_ablacion(), fig_aprendizaje(), fig_particion(), fig_wape_sesgo(),
             fig_horizonte_tipo_nivel(), fig_serie_total(), fig_bootstrap(), fig_tia(), fig_pronosticos_relaciones(),
             fig_interpretabilidad()]

### Resumen de resultados
Una sola tabla con las cifras que cita el texto de los capítulos V y VI. Si tus resultados difieren de los del documento (por ejemplo, porque la base sintética se generó de nuevo), basta con actualizar el texto con esta tabla.

In [ ]:
r = {}
r["transformacion"], r["exogenas"] = MODO_Y, ESPEC
for n in MODELOS + [CRO, EMP, EMP_OR]:
    r[f"WAPE prueba · {n}"] = TOTAL.loc[n, "WAPE"]
    r[f"Sesgo % · {n}"] = TOTAL.loc[n, "sesgo_total_pct"]
cb_ = comb.set_index("metodo")
r["WAPE prueba · Combinación"] = cb_.loc["Combinación RF + Croston", "WAPE_prueba"]
r["Dif. combinación − empresa (pp)"] = 100 * cb_.loc["Combinación RF + Croston", "dif_vs_empresa"]
r["IC95 combinación (pp)"] = f"[{100 * cb_.loc['Combinación RF + Croston', 'IC95_inf']:.1f}; {100 * cb_.loc['Combinación RF + Croston', 'IC95_sup']:.1f}]"
r["Cobertura intervalo 80 %"] = cobertura
for n in ["Random Forest", "Combinación RF + Croston", EMP, CRO]:
    fila = cb_.loc[n] if n in cb_.index else None
    if fila is not None:
        r[f"WAPE empresa total · {n}"] = fila.WAPE_empresa
        r[f"WAPE línea · {n}"] = fila.WAPE_linea
        r[f"WAPE cliente · {n}"] = fila.WAPE_cliente
r["Modelo SHAP"] = BOOST
r["Top 5 SHAP"] = ", ".join(NOM_VAR.get(k, k) for k in imp_shap.index[:5])
resumen = pd.Series(r, name="valor")
resumen.to_csv(DIR_TAB / "R00_resumen_resultados.csv", encoding="utf-8-sig")
print("Figuras:", sorted(p.name for p in DIR_FIG.glob("*.png")))
resumen.to_frame()

## Paso 13. Figuras y tablas de los anexos
Material de apoyo para los anexos: descomposición STL y autocorrelación de la venta total, pruebas por relación, series externas, venta observada frente a pronosticada, error por tipo de cliente y línea, escenario del semestre de prueba, SHAP por observación, LIME de tres relaciones, WAPE por horizonte y versiones del software. Como el Paso 12, todo se lee de disco.

In [ ]:
from statsmodels.tsa.stattools import acf as _acf, adfuller as _adf
from statsmodels.stats.diagnostic import acorr_ljungbox as _lb
from statsmodels.tsa.seasonal import STL as _STL

_serie = panel.groupby("mes")["ventas_usd"].sum()
_fechas = _serie.index.to_timestamp()
INI_PRUEBA = (pd.Period(FIN_VALIDACION, "M") + 1).to_timestamp()


def fig_stl_acf():
    y = 100 * _serie.values / _serie[_serie.index.year == 2023].mean()
    stl = _STL(y, period=12, robust=True).fit()
    fig = plt.figure(figsize=(ANCHO, 9.2 * CM))
    g = fig.add_gridspec(3, 2, width_ratios=[1.75, 1], hspace=0.28, wspace=0.32)
    for k, (comp, et, col) in enumerate([(stl.trend, "Tendencia", PALETA[0]), (stl.seasonal, "Estacionalidad", PALETA[2]),
                                         (stl.resid, "Residuo", GRISES[0])]):
        a = fig.add_subplot(g[k, 0])
        a.plot(_fechas, comp, color=col, lw=1.3)
        a.set_ylabel(et, fontsize=7.5)
        a.xaxis.set_major_locator(mdates.YearLocator())
        a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        if k < 2:
            a.tick_params(labelbottom=False)
        if k == 2:
            a.axhline(0, color=GRISES[1], lw=0.7)
    a = fig.add_subplot(g[:, 1])
    r = _acf(y, nlags=18)
    lim = 1.96 / np.sqrt(len(y))
    a.axhspan(-lim, lim, color=GRISES[2], alpha=0.35, lw=0)
    a.vlines(range(1, 19), 0, r[1:], color=PALETA[0], lw=1.3)
    a.axhline(0, color=GRISES[0], lw=0.8)
    a.set_xticks([1, 6, 12, 18])
    a.set_xlabel("Rezago (meses)")
    a.set_ylabel("Autocorrelación")
    return guardar(fig, "A_stl_acf")


def tabla_pruebas_por_relacion(min_meses=24, min_compras=12):
    filas = []
    for pid, g_ in panel.groupby("par_id"):
        y = g_.sort_values("mes")["ventas_usd"].values.astype(float)
        if len(y) >= min_meses and (y != 0).sum() >= min_compras and np.std(y) > 0:
            filas.append({"segmento": g_["segmento_sb"].iat[0], "adf_p": _adf(y, autolag="AIC")[1],
                          "lb12_p": _lb(y, lags=[12], return_df=True)["lb_pvalue"].iat[0]})
    d = pd.DataFrame(filas)
    res = d.groupby("segmento").agg(relaciones=("adf_p", "size"), pct_estacionarias_ADF=("adf_p", lambda v: 100 * (v < 0.05).mean()),
                                    pct_autocorrelacion_LB12=("lb12_p", lambda v: 100 * (v < 0.05).mean()))
    res.loc["Total"] = [len(d), 100 * (d.adf_p < 0.05).mean(), 100 * (d.lb12_p < 0.05).mean()]
    res.round(1).to_csv(DIR_TAB / "R27_pruebas_por_relacion.csv", encoding="utf-8-sig")
    return res.round(1)


NOM_EXT = {"ipc_general_var_a": "Inflación general (%)", "ipc_salud_var_a": "Inflación de salud (%)",
           "desempleo": "Desempleo (%)", "imae_var_a": "IMAE (var. anual, %)", "riesgo_pais": "Riesgo país (pb)",
           "tasa_activa": "Tasa activa (%)", "remesas_var_a": "Remesas (var. anual, %)", "dengue_log": "Dengue (log casos)",
           "petroleo_wti": "Petróleo WTI (USD/barril)", "oni": "Índice ONI (°C)"}


def fig_series_externas():
    z = exog.loc[pd.Period(PERIODO_INICIO, "M"):pd.Period(PERIODO_FIN, "M")]
    t = z.index.to_timestamp()
    fig, ax = plt.subplots(2, 5, figsize=(ANCHO, 6.4 * CM), sharex=True)
    for a, (v, n) in zip(ax.ravel(), NOM_EXT.items()):
        a.axvspan(INI_PRUEBA, t[-1], color=FONDO_PART["prueba"], lw=0)
        a.plot(t, z[v].values, color=PALETA[0] if v in ADMITIDAS else GRISES[1], lw=1.1)
        a.set_title(n, fontsize=6.4, loc="left", color=TINTA)
        a.tick_params(labelsize=6)
        a.xaxis.set_major_locator(mdates.YearLocator(2))
        a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout(h_pad=0.6, w_pad=0.4)
    return guardar(fig, "A_series_externas")


def fig_obs_pred():
    g_ = lambda v: np.sign(v) * np.log1p(np.abs(v))
    sel = [("p_comb", "Combinación RF + Croston"), ("p_rf", "Random Forest"), ("p_cro", CRO), (f"p_{EMP}", "Método de la empresa")]
    fig, ax = plt.subplots(1, 4, figsize=(ANCHO, 4.5 * CM), sharex=True, sharey=True)
    for a, (col, n) in zip(ax, sel):
        a.hexbin(g_(te.y), g_(te[col]), gridsize=28, bins="log", cmap=AZULES, mincnt=1, linewidths=0, extent=(-12, 14, -12, 14))
        a.plot([-12, 14], [-12, 14], color=PALETA[1], lw=0.8, ls="--")
        a.set_title(n, fontsize=6.6, loc="left", color=TINTA)
        a.set_xlabel("g(y) observado", fontsize=7)
        a.grid(False)
    ax[0].set_ylabel("g(ŷ) pronosticado", fontsize=7)
    fig.tight_layout(w_pad=0.4)
    return guardar(fig, "A_observado_pronosticado")


def fig_mapa_tipo_linea():
    top = te.groupby("sku", observed=True).y.apply(lambda v: v.abs().sum()).sort_values(ascending=False).head(12).index
    q = te[te.sku.isin(top)]
    filas = []
    for (t_, l_), d in q.groupby(["tratamiento", "sku"], observed=True):
        if d.par_id.nunique() >= 5 and d.y.abs().sum() > 0:
            filas.append({"tipo": t_, "linea": l_, "dif": 100 * (wape(d.y, d["p_rf"]) - wape(d.y, d[f"p_{EMP}"]))})
    mp = pd.DataFrame(filas).pivot(index="tipo", columns="linea", values="dif").reindex(columns=list(top))
    mp.round(1).to_csv(DIR_TAB / "R28_mapa_tipo_linea.csv", encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(ANCHO, 4.8 * CM))
    lim = np.nanpercentile(np.abs(mp.values), 95)
    im = ax.imshow(mp.values, cmap=DIVERGENTE, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(mp.shape[1]), mp.columns, fontsize=7)
    ax.set_yticks(range(mp.shape[0]), mp.index, fontsize=7)
    ax.grid(False)
    for (i, j), v in np.ndenumerate(mp.values):
        if not np.isnan(v):
            ax.text(j, i, f"{v:+.0f}".replace("-", "−"), ha="center", va="center", fontsize=6.2, color=TINTA)
    ax.set_xlabel("Las 12 líneas con mayor venta en la prueba (de mayor a menor)")
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("WAPE RF − empresa (pp)", fontsize=7)
    fig.tight_layout()
    return guardar(fig, "A_mapa_tipo_linea")


def fig_escenario():
    o = te[te.origen == pd.Period(FIN_VALIDACION, "M")]
    a = o.groupby("mes_objetivo")[["y", "p_rf", "p_comb", f"p_{EMP}", "p_cro"]].sum()
    base = a.y.mean()
    xs = [f"{MES_ES[m.month - 1]} {m.year}" for m in a.index]
    fig, ax = plt.subplots(figsize=(ANCHO, 5.4 * CM))
    ax.plot(xs, 100 * a.y / base, color=TINTA, marker="o", ms=3.5, lw=1.2, label="Observado")
    ax.plot(xs, 100 * a.p_rf / base, color=COLOR_MODELO["Random Forest"], marker="s", ms=3, label="Random Forest")
    ax.plot(xs, 100 * a.p_comb / base, color=COLOR_COMB, marker="^", ms=3, label="Combinación RF + Croston")
    ax.plot(xs, 100 * a[f"p_{EMP}"] / base, color=COLOR_EMP, ls="--", lw=1.1, label="Método de la empresa")
    ax.plot(xs, 100 * a.p_cro / base, color=GRISES[1], ls="-.", lw=1.3, label=CRO)
    ax.set_ylabel("Índice (media observada = 100)")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.13), fontsize=7)
    sem = (100 * (a.sum() / a.y.sum() - 1)).drop("y").round(1)
    sem.rename("desvio_semestral_pct").to_csv(DIR_TAB / "R29_escenario_semestre.csv", encoding="utf-8-sig")
    fig.tight_layout()
    return guardar(fig, "A_escenario_semestre")


def fig_shap_beeswarm():
    ruta = DIR_INT / "shap_muestra.pkl"
    if not ruta.exists():
        print("Sin shap_muestra.pkl (se crea en el Paso 11): se omite la figura.")
        return None
    o = pd.read_pickle(ruta)
    sv, Z = np.asarray(o["sv"]), o["Z"]
    orden = np.argsort(np.abs(sv).mean(0))[::-1][:12]
    fig, ax = plt.subplots(figsize=(ANCHO, 8 * CM))
    rng_ = np.random.default_rng(SEMILLA)
    for k, j in enumerate(orden[::-1]):
        x = pd.to_numeric(Z.iloc[:, j], errors="coerce")
        rk = x.rank(pct=True).fillna(0.5).values
        ax.scatter(sv[:, j] / 1e3, k + rng_.uniform(-0.3, 0.3, len(rk)), c=rk, cmap=DIVERGENTE, s=2.5, alpha=0.7, lw=0, rasterized=True)  # 36 000 puntos rasterizados: archivo vectorial liviano
    ax.set_yticks(range(len(orden)), [NOM_VAR.get(o["cols"][j], o["cols"][j]) for j in orden[::-1]], fontsize=7)
    ax.axvline(0, color=GRISES[0], lw=0.8)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xticks([-100, -10, -1, 0, 1, 10, 100, 1000], ["−100", "−10", "−1", "0", "1", "10", "100", "1000"])
    esc = "escala g(y)" if MODO_Y == "slog" else "miles USD, escala logarítmica simétrica"
    ax.set_xlabel(f"Contribución SHAP al pronóstico ({esc}) · {o['modelo']}")
    sm = plt.cm.ScalarMappable(cmap=DIVERGENTE)
    cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, ticks=[0, 1])
    cb.ax.set_yticklabels(["bajo", "alto"], fontsize=6.5)
    cb.set_label("Valor de la variable", fontsize=7)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return guardar(fig, "A_shap_observaciones")


def fig_lime_tres():
    casos = [c_ for c_ in ["suave", "intermitente", "irregular (lumpy)"] if c_ in set(lime_tab.caso)]
    fig, ax = plt.subplots(len(casos), 1, figsize=(ANCHO, 3.3 * len(casos) * CM), sharex=True)
    for k, (a, c_) in enumerate(zip(np.atleast_1d(ax), casos)):
        d = lime_tab[lime_tab.caso == c_].iloc[::-1]
        a.barh([regla_legible(r_) for r_ in d.regla], d.peso / 1e3, color=[PALETA[0] if v > 0 else PALETA[1] for v in d.peso], height=0.6)
        a.axvline(0, color=GRISES[0], lw=0.8)
        a.tick_params(axis="y", labelsize=6.6)
        a.grid(axis="y", visible=False)
        etiqueta(a, f"({'abc'[k]}) Relación {NOM_SEG.get(c_, c_).lower()}")
    np.atleast_1d(ax)[-1].set_xlabel("Efecto en el pronóstico a un mes (miles USD)")
    fig.tight_layout(h_pad=0.8)
    return guardar(fig, "A_lime_tres_casos")


_hh = ev[ev.grupo.str.startswith("h=")].assign(h=lambda d: d.grupo.str[2:].astype(int))
_hh.pivot(index="modelo", columns="h", values="WAPE").round(3).to_csv(DIR_TAB / "R30_wape_por_horizonte.csv", encoding="utf-8-sig")
import sys as _sys
import importlib.metadata as _md
_vers = [{"paquete": "Python", "version": _sys.version.split()[0]}]
for _p in ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "catboost", "statsmodels", "shap", "lime", "matplotlib"]:
    try:
        _vers.append({"paquete": _p, "version": _md.version(_p)})
    except Exception:
        _vers.append({"paquete": _p, "version": "no instalado"})
pd.DataFrame(_vers).to_csv(DIR_TAB / "R31_versiones.csv", index=False, encoding="utf-8-sig")
figuras_anexos = [fig_stl_acf(), fig_series_externas(), fig_obs_pred(), fig_mapa_tipo_linea(), fig_escenario(),
                  fig_shap_beeswarm(), fig_lime_tres()]
tabla_pruebas_por_relacion()
print("Figuras vectoriales (SVG):")
for _f in sorted(DIR_SVG.glob("*.svg")):
    print(f"  {_f.name}  ({_f.stat().st_size / 1024:,.0f} KB)")